### Full overview
1. context on all cell lines: vcc_context_profile_v3.py profiles the challenge lines' controls; vcc_build_effects_v3.py builds the public-data effect tables and writes contexts.json (make-config)
2. vcc_transfer_model_v3.py **train** trained on public dataset contexts to decide optimal weights
3. vcc_transfer_model_v3.py **predict** context from our 18,400 control cells are fed into this script to run inference, outputs the predicted profile of the control cell lines
4. vcc_generate_v3.py **generates** our 400 cells for submission, outputs prediction.h5ad -> vcc prep


Additional notes:
1. public datasets provide the contexts (e.g Orion, Tcell, Replogle, Tahoe)
2. they are generally pseudobulked and used to predict log changes and standard error
3. the 18,400 control cells are used to generate a more realistic distribution of the 400 submission cells
4. Arc H1 dataset is used to fine-tune the generator, not for training purposes

## 0. Setup: run this cell first, every session
Sets the folders, checks the data and tools, and defines three helpers:
- `run(cmd)` runs a short step now and shows its output here.
- `bg(name, cmd)` runs a long step in the background (tmux). It keeps going even if VS Code disconnects or this notebook closes.
- `status(name)` shows a background step's progress; it says FINISHED OK or STOPPED when done.

In [ ]:
import os, sys, shutil, subprocess, shlex
from pathlib import Path
V = Path.home() / 'vcc'
ENV = dict(CODE=str(V / 'code'), DATA=str(V / 'data'), LIB=str(V / 'lib'), LOGS=str(V / 'logs'),
           B='gs://vcc2026-data-mooneysad', PY=sys.executable)          # PY = this notebook's Python (your venv)
os.environ.update(ENV)
for d in ('CODE', 'LOGS', 'LIB'):
    os.makedirs(ENV[d], exist_ok=True)

def run(cmd):
    """Short step: run now (from the code folder), stream the output here, stop the cell on error."""
    p = subprocess.Popen(['bash', '-c', 'set -eo pipefail; cd "$CODE"\n' + cmd], env=os.environ,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}) - see the output above')

def bg(name, cmd):
    """Long step: run in tmux session `name`; output goes to ~/vcc/logs/<name>.log."""
    log, sh = Path(ENV['LOGS']) / f'{name}.log', Path(ENV['LOGS']) / f'{name}.sh'
    exports = ''.join(f'export {k}={shlex.quote(v)}\n' for k, v in ENV.items())
    sh.write_text(f'#!/bin/bash\n{exports}exec > {shlex.quote(str(log))} 2>&1\n'
                  f'trap \'echo "EXIT=$?"\' EXIT\nset -eo pipefail\ncd "$CODE"\n{cmd}\n')
    subprocess.run(['tmux', 'kill-session', '-t', name], stderr=subprocess.DEVNULL)
    subprocess.run(['tmux', 'new-session', '-d', '-s', name, 'bash', str(sh)], check=True)
    print(f'started "{name}" in the background. Check with: status("{name}")')

def status(name, n=20):
    alive = subprocess.run(['tmux', 'has-session', '-t', name], stderr=subprocess.DEVNULL).returncode == 0
    log = Path(ENV['LOGS']) / f'{name}.log'
    lines = log.read_text().splitlines() if log.exists() else []
    state = 'RUNNING' if alive else ('FINISHED OK' if lines and lines[-1] == 'EXIT=0' else 'STOPPED - read the log below')
    print(f'== {name}: {state}\n' + '\n'.join(lines[-n:]))

# ---- checks ----
need = ['data/gene_names.csv', 'data/pert_counts.csv', 'data/context_A.h5ad', 'data/context_B.h5ad', 'data/context_C.h5ad',
        'lib/orion/HCT116_pseudobulk.h5ad', 'lib/orion/HEK293T_pseudobulk.h5ad',
        'lib/tcell/TCELL_Stim48hr_effects.h5ad', 'lib/tcell/TCELL_Rest_effects.h5ad', 'lib/tcell/tcell_targets_effects.h5ad',
        'lib/h1/h1_pseudobulk.h5ad', 'lib/h1/h1_cells_oracle.h5ad']
missing = [f for f in need if not (V / f).exists()]
print('python:', sys.executable)
print('data files: all present' if not missing else f'MISSING (copy them from the bucket, runbook A8): {missing}')
print('tmux:', 'OK' if shutil.which('tmux') else 'MISSING -> in the terminal: sudo apt-get install -y tmux')
for m in ['numpy', 'pandas', 'scipy', 'h5py', 'anndata', 'sklearn']:
    try:
        __import__(m)
    except ImportError:
        print(f'package {m} MISSING -> in this cell: %pip install {m if m != "sklearn" else "scikit-learn"}')

## 1. Write the scripts to ~/vcc/code
Run these five cells after ANY edit to them: running a cell (re)writes that script file, and the steps below run the files.
Edit the code here; this notebook is the one file you keep.

#### vcc_common.py

In [ ]:
%%writefile ~/vcc/code/vcc_common.py
"""Shared helpers for the VCC 2026 v3 pipeline.

Imported by:
  vcc_context_profile_v3.py : read_names, alignment, select_controls, moments, save_profile
  vcc_transfer_model_v3.py  : Effects, PROGRAMS, load_profile, read_names, save_json, seed_for, shrink
  vcc_build_effects_v3.py   : everything below (writes the formats the other two read)

On-disk formats (one folder per public context, written by vcc_build_effects_v3.py)
  <dir>/lfc.npy              knockdowns x genes, float16 (memory-mapped), log2 fold change vs that context's controls
  <dir>/se.npy               same shape, standard error of the LFC (NaN = unknown)
  <dir>/targets.npy          knockdown names (row order)
  <dir>/genes.npy            gene names (column order)
  <dir>/meta.json            how the numbers were made (se_kind, lfc definition, legacy flag, ...)
  <dir>/control_profile.npz  baseline profile of the context's controls (same format as a profile .npz)
  <dir>/context.json         suggested name / background / platform for contexts.json
  A context is "legacy" when its SE is missing (no honest uncertainty); loading it then needs allow_legacy=True.

Profile .npz (written by save_profile): genes, meta (json string), and per-gene arrays aligned to `genes`:
  mean_cpm   mean of per-cell CPM (the scorer's > 5 CPM gene gate uses this)
  pooled_cpm ratio-of-sums CPM (sum of counts / total * 1e6), comparable across single-cell AND pseudobulk contexts
  log_mean   log1p(pooled_cpm)  <- the baseline the transfer model compares across contexts
  detection, var_cpm, se_mean (NaN when not computable, e.g. pseudobulk-only contexts)
  plus program_names / program_levels (mean log1p pooled CPM over each program's genes), n_cells, legacy_baseline.
  Genes a context did not measure are NaN.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

CONTROL = "non-targeting"

# Program gene sets. 'prefix:' entries are expanded against the gene list (ribosomal proteins, mitochondrial genes).
PROGRAMS = {
    "cc_S": "MCM5 PCNA TYMS FEN1 MCM2 MCM4 RRM1 UNG GINS2 MCM6 CDCA7 DTL PRIM1 UHRF1 HELLS RFC2 RPA2 NASP RAD51AP1 GMNN "
            "WDR76 SLBP CCNE2 UBR7 POLD3 MSH2 ATAD2 RAD51 RRM2 CDC45 CDC6 EXO1 TIPIN DSCC1 BLM CASP8AP2 USP1 CLSPN POLA1 "
            "CHAF1B BRIP1 E2F8".split(),
    "cc_G2M": "HMGB2 CDK1 NUSAP1 UBE2C BIRC5 TPX2 TOP2A NDC80 CKS2 NUF2 CKS1B MKI67 TMPO CENPF TACC3 SMC4 CCNB2 CKAP2L "
              "CKAP2 AURKB BUB1 KIF11 ANP32E TUBB4B GTSE1 KIF20B HJURP CDCA3 CDC20 TTK CDC25C KIF2C RANGAP1 NCAPD2 DLGAP5 "
              "CDCA2 CDCA8 ECT2 KIF23 HMMR AURKA PSRC1 ANLN LBR CKAP5 CENPE CTCF NEK2 G2E3 GAS2L3 CBX5 CENPA".split(),
    "stress": "HSPA1A HSPA1B HSPA8 HSPH1 DNAJB1 HSP90AA1 FOS JUN EGR1 ATF3 IER2 DUSP1 DDIT3 ATF4".split(),
    "interferon": "ISG15 IFI6 IFIT1 IFIT2 IFIT3 MX1 OAS1 IFI44L STAT1 IRF7 IFITM1 BST2".split(),
    "hypoxia_glyc": "VEGFA CA9 BNIP3 PGK1 LDHA SLC2A1 ENO1 ALDOA NDRG1 P4HA1 PFKFB3 HK2".split(),
    "biogenesis": "NPM1 NCL NOP56 NOP58 FBL RRP9 PA2G4 CAD ODC1 DKC1 NOP16 POLR1B".split(),
    "ribosome": ["prefix:RPL", "prefix:RPS"],
    "mito": ["prefix:MT-"],
}
_PREFIX_EXCLUDE = ("RPL3L", "RPS4Y")


# ---------------------------------------------------------------- small utilities
def read_names(path, col):
    """One column of a CSV as a numpy string array (first column if `col` is absent)."""
    df = pd.read_csv(path)
    return np.asarray(df[col] if col in df.columns else df.iloc[:, 0], dtype=str)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=1, default=_json_default))


def seed_for(seed, *keys):
    """Deterministic seed from (seed, keys...): stable across runs and machines (unlike Python's hash())."""
    h = hashlib.sha256(repr((int(seed),) + tuple(str(k) for k in keys)).encode()).digest()
    return int.from_bytes(h[:8], "little")


def shrink(lfc, se, cap=8.0, tau=4.0, missing_se_factor=0.5):
    """Signal-to-noise shrinkage (same rule as M1): lfc * snr^2 / (snr^2 + tau).
    An effect at 2 SE keeps 50%, at 4 SE keeps 80%. Missing SE -> fixed factor (legacy behaviour). Clipped to +-cap."""
    lfc = np.asarray(lfc, dtype=np.float64)
    se = np.asarray(se, dtype=np.float64) if se is not None else np.full_like(lfc, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        snr2 = np.where(np.isfinite(se) & (se > 0), (lfc / se) ** 2, np.nan)
        out = np.where(np.isfinite(snr2), lfc * snr2 / (snr2 + tau), lfc * missing_se_factor)
    out = np.where(np.isfinite(lfc), np.clip(out, -cap, cap), np.nan)
    return out.astype(np.float32)


# ---------------------------------------------------------------- gene alignment and programs
def alignment(source_names, genes):
    """For each of OUR genes, its column in the source (-1 = the source did not measure it)."""
    pos = {g: i for i, g in enumerate(np.asarray(source_names, dtype=str))}
    idx = np.array([pos.get(g, -1) for g in np.asarray(genes, dtype=str)], dtype=np.int64)
    if (idx >= 0).sum() == 0:
        raise ValueError("no genes in common with the provided gene list")
    return idx


def program_gene_index(genes):
    """{program: indices into genes}."""
    genes = np.asarray(genes, dtype=str); pos = {g: i for i, g in enumerate(genes)}; out = {}
    for name, members in PROGRAMS.items():
        idx = []
        for m in members:
            if m.startswith("prefix:"):
                p = m.split(":", 1)[1]
                idx += [i for i, g in enumerate(genes) if g.startswith(p) and not g.startswith(_PREFIX_EXCLUDE)]
            elif m in pos:
                idx.append(pos[m])
        out[name] = np.array(sorted(set(idx)), dtype=np.int64)
    return out


def program_levels(pooled_cpm, genes):
    """Mean log1p(pooled CPM) over each program's measured genes (NaN if < 3 measured)."""
    lv = []
    for name, idx in program_gene_index(genes).items():
        v = np.log1p(pooled_cpm[idx]) if len(idx) else np.array([])
        v = v[np.isfinite(v)]
        lv.append(float(v.mean()) if len(v) >= 3 else np.nan)
    return np.array(list(PROGRAMS)), np.array(lv, dtype=np.float32)


# ---------------------------------------------------------------- profiles from cells
def select_controls(a, obs_col, control=CONTROL, all_controls=False, max_cells=20000, seed=0):
    """Row indices of control cells. If `obs_col` is absent every row is treated as a control (e.g. context_A.h5ad).
    Without all_controls, at most `max_cells` controls (random, seeded) are used."""
    if obs_col in a.obs.columns:
        rows = np.flatnonzero(a.obs[obs_col].astype(str).values == control)
    else:
        rows = np.arange(a.n_obs)
    if len(rows) == 0:
        raise ValueError(f"no rows with {obs_col} == {control!r}")
    if not all_controls and len(rows) > max_cells:
        rows = np.sort(np.random.default_rng(seed).choice(rows, max_cells, replace=False))
    return np.sort(rows)


def _matrix(a, layer):
    if layer is None:
        return a.X
    if layer in a.layers:
        return a.layers[layer]
    raise KeyError(f"layer {layer!r} not found")


def moments(a, rows, idx, layer=None, block=512):
    """Per-gene statistics over `rows` (cells), streamed `block` cells at a time. Library size = sum over OUR genes.
    Returns arrays aligned to our gene list (NaN where the source lacks the gene)."""
    X = _matrix(a, layer); G = len(idx); ok = idx >= 0; cols = idx[ok]
    s_cnt = np.zeros(ok.sum()); s_cpm = np.zeros(ok.sum()); s_cpm2 = np.zeros(ok.sum()); s_det = np.zeros(ok.sum())
    libs = []
    for b in range(0, len(rows), block):
        r = np.asarray(rows[b:b + block])
        M = X[r]
        M = (M.tocsr() if sp.issparse(M) else sp.csr_matrix(np.asarray(M)))[:, cols].astype(np.float64)
        lib = np.asarray(M.sum(1)).ravel(); libs.append(lib)
        C = sp.diags(1e6 / np.maximum(lib, 1)) @ M
        s_cnt += np.asarray(M.sum(0)).ravel(); s_cpm += np.asarray(C.sum(0)).ravel()
        s_cpm2 += np.asarray(C.multiply(C).sum(0)).ravel(); s_det += np.asarray((M > 0).sum(0)).ravel()
    n = len(rows); lib = np.concatenate(libs) if libs else np.array([0.0])

    def full(v):
        out = np.full(G, np.nan, dtype=np.float32); out[ok] = v; return out
    mean = s_cpm / n; var = np.maximum(s_cpm2 / n - mean ** 2, 0)
    pooled = s_cnt / max(s_cnt.sum(), 1) * 1e6
    stats = dict(mean_cpm=full(mean), pooled_cpm=full(pooled), log_mean=full(np.log1p(pooled)),
                 detection=full(s_det / n), var_cpm=full(var), se_mean=full(np.sqrt(var / n)),
                 n_cells=np.int64(n), median_library=np.float64(np.median(lib)), measured=ok.copy(),
                 legacy_baseline=np.bool_(False))
    return stats


def finalise_profile(stats, genes):
    """Add program levels and the scorer gate flag."""
    names, lv = program_levels(stats["pooled_cpm"], genes)
    stats["program_names"] = names; stats["program_levels"] = lv
    stats["scored"] = np.nan_to_num(stats["mean_cpm"], nan=0) > 5
    return stats


def save_profile(output, genes, stats, meta=None):
    """Write a profile .npz (program levels added if missing)."""
    stats = dict(stats)
    if "program_levels" not in stats:
        stats = finalise_profile(stats, genes)
    out = Path(output); out.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out, genes=np.asarray(genes, dtype=str), meta=json.dumps(meta or {}, default=_json_default), **stats)
    return out


def load_profile(path, genes):
    """Profile re-aligned to `genes`. Per-gene arrays: NaN (or False) where missing. Returns a plain dict."""
    z = np.load(path, allow_pickle=False)
    pg = z["genes"].astype(str); src = alignment(pg, genes); ok = src >= 0
    out = {}
    for k in z.files:
        if k in ("genes", "meta"):
            continue
        v = z[k]
        if v.ndim == 1 and len(v) == len(pg) and k not in ("program_names", "program_levels", "control_obs_names"):
            if v.dtype == bool:
                w = np.zeros(len(genes), bool); w[ok] = v[src[ok]]
            else:
                w = np.full(len(genes), np.nan, dtype=np.float32); w[ok] = v[src[ok]]
            out[k] = w
        else:
            out[k] = v.item() if v.ndim == 0 else v
    out["genes"] = np.asarray(genes, dtype=str)
    out["meta"] = json.loads(str(z["meta"])) if "meta" in z.files else {}
    out["legacy_baseline"] = bool(out.get("legacy_baseline", False))
    if "program_levels" not in out:
        out["program_names"], out["program_levels"] = program_levels(np.expm1(out["log_mean"]), genes)
    out["program_levels"] = np.nan_to_num(np.asarray(out["program_levels"], dtype=np.float32), nan=0.0)
    for k in ("log_mean", "mean_cpm"):
        if k not in out:
            raise KeyError(f"profile {path} lacks {k}")
    return out


# ---------------------------------------------------------------- effect tables
class Effects:
    """One public context's effect table: LFC and SE per (knockdown, gene), memory-mapped from disk.
    get(kd, gene_idx) -> (lfc, se) float32 arrays for OUR gene indices (NaN = not measured)."""

    def __init__(self, path, genes, allow_legacy=False):
        p = Path(path); self.path = p
        self.meta = json.loads((p / "meta.json").read_text()) if (p / "meta.json").exists() else {}
        self.legacy = bool(self.meta.get("legacy", False))
        if self.legacy and not allow_legacy:
            raise ValueError(f"{p} is a legacy effect table (no usable SE); pass --allow-legacy-effects to use it")
        self.lfc = np.load(p / "lfc.npy", mmap_mode="r")
        self.se = np.load(p / "se.npy", mmap_mode="r") if (p / "se.npy").exists() else None
        self.targets = [str(t) for t in np.load(p / "targets.npy")]
        self.pos = {t: i for i, t in enumerate(self.targets)}
        self.col = alignment(np.load(p / "genes.npy").astype(str), genes)    # our gene -> stored column
        self.n_genes = len(genes)

    def get(self, kd, gi):
        gi = np.asarray(gi, dtype=np.int64).ravel()
        lfc = np.full(len(gi), np.nan, np.float32); se = np.full(len(gi), np.nan, np.float32)
        r = self.pos.get(kd)
        if r is None:
            return lfc, se
        c = self.col[gi]; ok = c >= 0
        if ok.any():
            row = np.asarray(self.lfc[r], dtype=np.float32); lfc[ok] = row[c[ok]]
            if self.se is not None:
                srow = np.asarray(self.se[r], dtype=np.float32); se[ok] = srow[c[ok]]
        return lfc, se

    def __repr__(self):
        return f"Effects({self.path.name}: {len(self.targets):,} knockdowns, legacy={self.legacy})"


def write_effects(out, genes, targets, rows_iter, meta, n_rows):
    """Stream (k, lfc_row, se_row) into an effect-table folder. Rows arrive in `targets` order."""
    from numpy.lib.format import open_memmap
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    L = open_memmap(out / "lfc.npy", mode="w+", dtype=np.float16, shape=(n_rows, len(genes)))
    S = open_memmap(out / "se.npy", mode="w+", dtype=np.float16, shape=(n_rows, len(genes)))
    done = 0
    for k, lfc, se in rows_iter:
        L[k] = np.clip(lfc, -60000, 60000); S[k] = np.clip(se, 0, 60000); done += 1
    L.flush(); S.flush(); del L, S
    np.save(out / "targets.npy", np.asarray(targets, dtype=str)); np.save(out / "genes.npy", np.asarray(genes, dtype=str))
    save_json(out / "meta.json", dict(meta, n_targets=int(n_rows), n_genes=int(len(genes))))
    assert done == n_rows, (done, n_rows)
    return out

#### vcc_build_effects_v3.py
%%writefile the script, then run orion ×2, tcell ×2, make-config

In [ ]:
%%writefile ~/vcc/code/vcc_build_effects_v3.py
"""Build the v3 inputs from our public data: effect tables, control profiles and contexts.json.

  orion       Orion per-knockdown sums (<line>_pseudobulk.h5ad)  -> effect folder (LFC + cell-level SE) + control profile
  h1          Arc H1 sums (h1_pseudobulk.h5ad, same layout)      -> same as orion (held-out validation context)
  tcell       T-cell effect table (TCELL_<cond>_effects.h5ad)   -> effect folder (LFC + donor-level SE) + control profile
  make-config effect folders                                     -> contexts.json for vcc_transfer_model_v3.py

Every effect folder also gets context.json (name / background / platform), so make-config needs no typing.

Colab
  !python vcc_build_effects_v3.py orion --pseudobulk /content/drive/MyDrive/vcc/orion/HCT116_pseudobulk.h5ad \
        --genes "$DATA/gene_names.csv" --name ORION_HCT116 --background HCT116 --out $LIB/effects/ORION_HCT116
  !python vcc_build_effects_v3.py tcell --effects $LIB/tcell/effects/TCELL_Stim48hr_effects.h5ad \
        --genes "$DATA/gene_names.csv" --name TCELL_Stim48hr --priority 1 --out $LIB/effects/TCELL_Stim48hr
  !python vcc_build_effects_v3.py make-config --genes "$DATA/gene_names.csv" --out $LIB/contexts.json \
        --dirs $LIB/effects/ORION_HCT116 $LIB/effects/ORION_HEK293T $LIB/effects/TCELL_Stim48hr $LIB/effects/TCELL_Rest
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import h5py
import numpy as np
import scipy.sparse as sp

from vcc_common import CONTROL, alignment, read_names, save_json, save_profile, write_effects

try:
    from anndata.io import read_elem, sparse_dataset
except ImportError:
    from anndata.experimental import read_elem, sparse_dataset

PSEUDO = 0.5


def reader(f, key):
    """rows(list of int) -> dense float64 block, for dense or sparse h5ad matrices."""
    node = f[key]
    if isinstance(node, h5py.Group):
        ds = sparse_dataset(node)
        return lambda r: np.asarray(ds[np.asarray(r)].toarray(), dtype=np.float64)
    return lambda r: np.asarray(node[np.asarray(r), :], dtype=np.float64)


def names_of(df, col="target_gene"):
    return np.asarray(df[col] if col in df.columns else df.index, dtype=str)


def context_file(out, name, background, platform, priority, kind):
    save_json(Path(out) / "context.json", dict(name=name, background=background, platform=platform,
                                             source_priority=priority, kind=kind))


# ---------------------------------------------------------------- Orion / H1 (per-knockdown sums)
def cmd_sums(a, kind):
    genes = read_names(a.genes, "gene_name"); out = Path(a.out)
    with h5py.File(a.pseudobulk, "r") as f:
        obs = read_elem(f["obs"]); var = read_elem(f["var"])
        control = a.control
        if control is None:
            control = str(read_elem(f["uns/control_label"])) if "uns/control_label" in f else CONTROL
        names = names_of(obs); n = obs["n_cells"].values.astype(np.float64)
        src = alignment(var.index.astype(str), genes); ok = src >= 0
        if "measured" in var.columns:
            meas = np.zeros(len(genes), bool); meas[ok] = var["measured"].values.astype(bool)[src[ok]]
        else:
            meas = ok.copy()
        rc, r1, r2, re = (reader(f, f"layers/{k}") for k in ("sum_counts", "sum_cpm", "sum_cpm2", "n_expr"))
        ci = int(np.flatnonzero(names == control)[0])

        def aligned(block):
            outb = np.full((block.shape[0], len(genes)), np.nan); outb[:, ok] = block[:, src[ok]]; return outb
        c_cnt, c_cpm, c_cpm2, c_det = (aligned(r([ci]))[0] for r in (rc, r1, r2, re))
        nc = n[ci]
        c_pool = c_cnt / np.nansum(np.where(meas, c_cnt, 0)) * 1e6
        c_mean = c_cpm / nc; c_var = np.maximum(c_cpm2 / nc - c_mean ** 2, 0)
        c_serel2 = (np.sqrt(c_var / nc) / (c_mean + PSEUDO)) ** 2
        keep = np.flatnonzero((names != control) & (n >= a.min_cells))
        targets = names[keep]
        # duplicate names (e.g. several guides): keep the row with most cells
        order = np.lexsort((-n[keep], targets)); first = np.r_[True, targets[order][1:] != targets[order][:-1]]
        keep = np.sort(keep[order][first]); targets = names[keep]
        print(f"{kind}: {len(keep):,} knockdowns with >= {a.min_cells} cells; control '{control}' with {int(nc):,} cells; "
              f"{int(meas.sum()):,}/{len(genes):,} genes measured")

        def rows():
            for b in range(0, len(keep), a.chunk):
                r = keep[b:b + a.chunk]
                cnt, cpm, cpm2 = (aligned(x(r)) for x in (rc, r1, r2))
                nk = n[r][:, None]
                pool = cnt / np.maximum(np.nansum(np.where(meas, cnt, 0), 1, keepdims=True), 1) * 1e6
                lfc = np.log2((pool + PSEUDO) / (c_pool + PSEUDO))
                mean = cpm / nk; var_ = np.maximum(cpm2 / nk - mean ** 2, 0)
                se = np.sqrt((np.sqrt(var_ / nk) / (mean + PSEUDO)) ** 2 + c_serel2) / np.log(2)
                lfc[:, ~meas] = np.nan; se[:, ~meas] = np.nan
                for k in range(len(r)):
                    yield b + k, lfc[k], se[k]
                print(f"  {min(b + a.chunk, len(keep)):,}/{len(keep):,}", end="\r", flush=True)
        write_effects(out, genes, targets, rows(), dict(
            kind=kind, source=str(Path(a.pseudobulk).resolve()), control_label=control, min_cells=a.min_cells,
            lfc="log2 ratio of pooled CPM (sum of counts / total over our genes), pseudocount 0.5 CPM",
            se_kind="cell_level (delta method from per-cell CPM spread; overconfident vs replicate SE)", legacy=False),
            len(keep))
    prof = dict(mean_cpm=np.where(meas, c_mean, np.nan).astype(np.float32),
                pooled_cpm=np.where(meas, c_pool, np.nan).astype(np.float32),
                detection=np.where(meas, c_det / nc, np.nan).astype(np.float32),
                var_cpm=np.where(meas, c_var, np.nan).astype(np.float32),
                se_mean=np.where(meas, np.sqrt(c_var / nc), np.nan).astype(np.float32),
                measured=meas, n_cells=np.int64(nc), legacy_baseline=np.bool_(False))
    prof["log_mean"] = np.log1p(prof["pooled_cpm"])
    save_profile(out / "control_profile.npz", genes, prof, dict(source=str(a.pseudobulk), kind=kind, baseline="controls row"))
    context_file(out, a.name, a.background, a.platform, a.priority, kind)
    print(f"\nwrote {out}  (lfc.npy, se.npy, control_profile.npz, context.json)")


# ---------------------------------------------------------------- T cells (already an effect table)
def cmd_tcell(a):
    genes = read_names(a.genes, "gene_name"); out = Path(a.out)
    with h5py.File(a.effects, "r") as f:
        obs = read_elem(f["obs"]); var = read_elem(f["var"])
        names = names_of(obs); src = alignment(var.index.astype(str), genes); ok = src >= 0
        meas = np.zeros(len(genes), bool)
        meas[ok] = var["measured"].values.astype(bool)[src[ok]] if "measured" in var.columns else True
        ctrl = np.full(len(genes), np.nan); ctrl[ok] = var["ctrl_cpm"].values.astype(float)[src[ok]]
        rx = reader(f, "X"); rs = reader(f, "layers/se") if "layers/se" in f else None
        ncell = obs["n_cells"].values if "n_cells" in obs.columns else np.ones(len(names))
        order = np.lexsort((-ncell, names)); first = np.r_[True, names[order][1:] != names[order][:-1]]
        keep = np.sort(order[first]); keep = keep[names[keep] != CONTROL]; targets = names[keep]
        print(f"tcell: {len(keep):,} knockdowns, {int(meas.sum()):,}/{len(genes):,} genes measured, SE: {rs is not None}")

        def rows():
            for b in range(0, len(keep), a.chunk):
                r = keep[b:b + a.chunk]
                L = np.full((len(r), len(genes)), np.nan); S = np.full_like(L, np.nan)
                L[:, ok] = rx(r)[:, src[ok]]
                if rs is not None:
                    S[:, ok] = rs(r)[:, src[ok]]
                L[:, ~meas] = np.nan; S[:, ~meas] = np.nan
                for k in range(len(r)):
                    yield b + k, L[k], S[k]
                print(f"  {min(b + a.chunk, len(keep)):,}/{len(keep):,}", end="\r", flush=True)
        write_effects(out, genes, targets, rows(), dict(
            kind="tcell", source=str(Path(a.effects).resolve()),
            lfc="replicate-paired log2 ratio of pseudobulk CPM (tcell_build_effects.py)",
            se_kind="donor_level (limma-style moderated)", legacy=rs is None), len(keep))
    pooled = np.where(meas, ctrl, np.nan).astype(np.float32)
    prof = dict(mean_cpm=pooled, pooled_cpm=pooled, log_mean=np.log1p(pooled),
                detection=np.full(len(genes), np.nan, np.float32), var_cpm=np.full(len(genes), np.nan, np.float32),
                se_mean=np.full(len(genes), np.nan, np.float32), measured=meas, n_cells=np.int64(0),
                legacy_baseline=np.bool_(True))     # baseline from pseudobulk, not single cells
    save_profile(out / "control_profile.npz", genes, prof, dict(source=str(a.effects), kind="tcell",
                 baseline="var['ctrl_cpm'] (pseudobulk; mean_cpm == pooled_cpm)"))
    context_file(out, a.name, a.background, a.platform, a.priority, "tcell")
    print(f"\nwrote {out}")


# ---------------------------------------------------------------- contexts.json
def cmd_config(a):
    ctx = []
    for d in a.dirs:
        d = Path(d); c = json.loads((d / "context.json").read_text())
        ctx.append(dict(name=c["name"], background=c["background"], platform=c["platform"],
                        source_priority=c.get("source_priority", 0), effects=str(d.resolve()),
                        profile=str((d / "control_profile.npz").resolve())))
    bgs = {}
    for c in ctx:
        bgs.setdefault(c["background"], []).append(c["name"])
    if len(bgs) < 2:
        raise SystemExit("need at least two biological backgrounds")
    cfg = dict(genes=str(Path(a.genes).resolve()), effect_cap=a.effect_cap, ridge_alphas=[10, 100, 1000, 10000], contexts=ctx)
    save_json(a.out, cfg)
    print(json.dumps(cfg, indent=1)); print("backgrounds (holdout names):", sorted(bgs))


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    for kind, bg, plat in (("orion", None, "orion"), ("h1", "H1", "flex")):
        s = sub.add_parser(kind); s.add_argument("--pseudobulk", required=True); s.add_argument("--genes", required=True)
        s.add_argument("--out", required=True); s.add_argument("--name", required=True)
        s.add_argument("--background", required=bg is None, default=bg); s.add_argument("--platform", default=plat)
        s.add_argument("--priority", type=int, default=0); s.add_argument("--control")
        s.add_argument("--min-cells", type=int, default=20); s.add_argument("--chunk", type=int, default=256)
    t = sub.add_parser("tcell"); t.add_argument("--effects", required=True); t.add_argument("--genes", required=True)
    t.add_argument("--out", required=True); t.add_argument("--name", required=True)
    t.add_argument("--background", default="CD4_T"); t.add_argument("--platform", default="flex")
    t.add_argument("--priority", type=int, default=0); t.add_argument("--chunk", type=int, default=256)
    m = sub.add_parser("make-config"); m.add_argument("--dirs", nargs="+", required=True)
    m.add_argument("--genes", required=True); m.add_argument("--out", required=True)
    m.add_argument("--effect-cap", type=float, default=8.0)
    a = ap.parse_args()
    {"orion": lambda: cmd_sums(a, "orion"), "h1": lambda: cmd_sums(a, "h1"),
     "tcell": lambda: cmd_tcell(a), "make-config": lambda: cmd_config(a)}[a.cmd]()

#### vcc_context_profile_v3.py
Profiles for lines A, B, C

In [ ]:
%%writefile ~/vcc/code/vcc_context_profile_v3.py
"""Profile an unseen line using ALL available raw-count control cells.

Transfer context uses ratio-of-sums CPM (bulk_cpm), not mean per-cell CPM.
Both are retained: mean_cpm/var_cpm/se_mean keep their original per-cell meaning
for DE diagnostics and the existing cell generator. This does not convert LFCs.

Example: python vcc_context_profile_v3.py --counts context_A.h5ad --all-controls
         --genes gene_names.csv --out profiles/A.npz
"""
import argparse
from pathlib import Path

import anndata as ad
import numpy as np

from vcc_common import alignment, moments, program_levels, read_names, save_profile, select_controls


def profile(counts, genes_csv, output, layer=None, obs_col="target_gene",
            control="non-targeting", all_controls=False, block=512):
    genes = read_names(genes_csv, "gene_name")
    a = ad.read_h5ad(counts, backed="r")
    try:
        idx = alignment(a.var_names.astype(str), genes)
        rows = select_controls(a, obs_col, control, all_controls)
        stats = moments(a, rows, idx, layer, block)
        # Explicit transfer representation. Do not overwrite mean_cpm/log_mean:
        # the DE scorer and the cell generator still need per-cell mean CPM.
        stats["baseline_cpm"] = stats["bulk_cpm"].copy()
        stats["baseline_log"] = np.log1p(stats["bulk_cpm"])
        stats["baseline_convention"] = np.array("ratio_of_sums_cpm")
        stats["baseline_program_levels"], stats["baseline_program_coverage"] = program_levels(genes, stats["bulk_cpm"])
        finite = np.isfinite(stats["baseline_log"])
        centre = float(stats["baseline_log"][finite].mean())
        stats["baseline_log_centered"] = stats["baseline_log"] - centre
        # These identifiers permit auditing overlap when making synthetic predictions.
        stats["control_obs_names"] = np.asarray(a.obs_names[rows], dtype=str)
        save_profile(output, genes, stats, dict(counts=str(Path(counts).resolve()), layer=layer,
                     obs_col=obs_col, control=control, all_controls=all_controls,
                     transfer_baseline_convention="ratio_of_sums_cpm",
                     transfer_baseline_field="bulk_cpm",
                     per_cell_mean_field="mean_cpm",
                     baseline_log_center=centre,
                     library_denominator="sum over the provided gene_names.csv axis"))
    finally:
        a.file.close()
    print(f"Saved {output}: {len(rows):,} controls, {len(genes):,} aligned genes")


if __name__ == "__main__":
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--counts", required=True)
    p.add_argument("--genes", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--layer")
    p.add_argument("--obs-col", default="target_gene")
    p.add_argument("--control", default="non-targeting")
    p.add_argument("--all-controls", action="store_true")
    p.add_argument("--block", type=int, default=512)
    a = p.parse_args()
    profile(a.counts, a.genes, a.out, a.layer, a.obs_col, a.control, a.all_controls, a.block)

#### vcc_transfer_model_v3.py
Hold-out tests → final train → predict

In [ ]:
%%writefile ~/vcc/code/vcc_transfer_model_v3.py
"""Context-conditioned transfer, with strict biological-background holdouts.

One training observation = (target context, knockdown, output gene).
Inputs = allowed source effects + untreated source/target context (never target effects).
Output = predicted log2 fold change for that gene in the target context.

Models: ridge residual over mean source effect; optional neural mixture/residual.
No independently fitted SVD embeddings; sources and missingness are explicit.

Review revision 1: retrain older checkpoints. Defaults: bulk context baselines,
global context features disabled, no platform feature, 50% source-signal training
sampling, capped label-precision weights, and trained-arity subset inference.
CLI ablations: --line-features, --signal-fraction 0, --no-label-weighting.
Legacy ctrl_cpm is accepted as bulk ONLY with verified baseline_convention metadata
or a context config declaration: "baseline_convention": "ratio_of_sums_cpm".
This changes context features, not the definition of existing effect-table labels.
"""
from __future__ import annotations

import argparse
import json
from itertools import combinations
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from vcc_common import Effects, PROGRAMS, load_profile, program_levels, read_names, save_json, seed_for, shrink

SRC_NAMES = ["source_lfc", "source_se", "se_missing", "source_shrunk_lfc",
             "source_baseline", "baseline_difference", "source_knockdown_gene_baseline",
             "knockdown_gene_baseline_difference", "source_ontarget_lfc", "ontarget_missing",
             "legacy_effect", "context_pearson",
             "agreement", "legacy_source_baseline"] + ["program_difference_" + k for k in PROGRAMS]
TGT_NAMES = ["target_baseline", "target_knockdown_gene_baseline", "knockdown_gene_missing",
             "legacy_target_baseline"] + ["target_program_" + k for k in PROGRAMS]


def transfer_profile(profile, mode="bulk", declared_convention=None, label="profile"):
    """Select a verified context estimator; never relabel mean_cpm as bulk silently.

    Existing canonical profiles already store bulk_cpm. A legacy table's ctrl_cpm
    lives in mean_cpm for compatibility, so its provenance must be declared explicitly.
    New fields are derived after named-gene alignment; saved derived arrays are not trusted.
    """
    p = dict(profile)
    convention = declared_convention or str(p.get("baseline_convention", ""))
    if mode == "bulk":
        if "bulk_cpm" in p:
            base = p["bulk_cpm"]
        elif convention == "ratio_of_sums_cpm":
            base = p["mean_cpm"]
        else:
            raise ValueError(f"{label}: no verified ratio-of-sums baseline. Supply bulk_cpm or verify "
                             "the original ctrl_cpm builder and set baseline_convention='ratio_of_sums_cpm' "
                             "in this context's config. --allow-legacy-effects does not establish baseline provenance.")
    elif mode == "per_cell":
        if bool(p.get("legacy_baseline", False)) and convention != "arithmetic_mean_per_cell_cpm":
            raise ValueError(f"{label}: legacy baseline is not a verified per-cell CPM mean")
        base = p["mean_cpm"]
    else:
        raise ValueError("baseline mode must be bulk or per_cell")
    base = np.asarray(base, dtype=float)
    if not np.isfinite(base).any() or np.any(base[np.isfinite(base)] < 0) or np.isinf(base).any():
        raise ValueError(f"{label}: invalid context baseline")
    p["baseline_cpm"] = base
    p["baseline_log"] = np.log1p(base)
    p["baseline_program_levels"], _ = program_levels(p["genes"], base)
    p["baseline_mode"] = mode
    p["legacy_baseline"] = np.array(False)  # Selected estimator has now been verified.
    return p


def centred_similarity(a, b):
    valid = np.isfinite(a) & np.isfinite(b)
    if valid.sum() < 2:
        return 0.0
    x, y = np.asarray(a[valid], float), np.asarray(b[valid], float)
    x -= x.mean(); y -= y.mean()
    denom = np.linalg.norm(x)*np.linalg.norm(y)
    return float(np.clip(np.dot(x, y)/denom, -1, 1)) if denom > 1e-12 else 0.0


class Context:
    def __init__(self, spec, genes, allow_legacy, baseline_mode="bulk"):
        self.name, self.background = spec["name"], spec["background"]
        self.platform = spec.get("platform", "unknown")
        self.priority = spec.get("source_priority", 0)
        self.effects = Effects(spec["effects"], genes, allow_legacy)
        self.profile = load_profile(spec.get("profile", str(Path(spec["effects"]) / "control_profile.npz")), genes)
        declaration = spec.get("baseline_convention", self.effects.meta.get("baseline_convention"))
        self.profile = transfer_profile(self.profile, baseline_mode, declaration, self.name)
        self.spec = spec


def load_contexts(config, genes, excluded=(), allow_legacy=False, baseline_mode="bulk"):
    # Filter BEFORE opening any effect arrays; a held-out context is not accidentally consulted.
    return [Context(s, genes, allow_legacy, baseline_mode) for s in config["contexts"] if s["background"] not in excluded]


def choose_sources(contexts, target_background=None):
    """At most one state per biological background; avoids double-counting T-cell states."""
    groups = {}
    for c in contexts:
        if c.background == target_background:
            continue
        if c.background not in groups or (c.priority, c.name) > (groups[c.background].priority, groups[c.background].name):
            groups[c.background] = c
    return sorted(groups.values(), key=lambda c: c.name)


def generic_signatures(contexts, train_targets, genes, cap):
    out = {}
    gi = np.arange(len(genes))
    for c in contexts:
        total, count = np.zeros(len(genes)), np.zeros(len(genes))
        for k in train_targets:
            if k not in c.effects.pos:
                continue
            l, se = c.effects.get(k, gi)
            sh = shrink(l, se, cap=cap)
            ok = np.isfinite(sh)
            total[ok] += sh[ok]; count[ok] += 1
        out[c.name] = np.divide(total, count, out=np.zeros_like(total), where=count > 0).astype(np.float32)
    return out


def features(target_profile, target_platform, sources, kd, gi, genes, generics, cap=8.0, line_features=False):
    """This function has no target outcome argument. All subset features rebuilt here."""
    gi = np.asarray(gi, int)
    gp = {g: i for i, g in enumerate(genes)}
    ki = gp.get(kd)
    if "baseline_mode" not in target_profile:
        target_profile = transfer_profile(target_profile)
    tb = target_profile["baseline_log"]
    tk = tb[ki] if ki is not None and np.isfinite(tb[ki]) else 0.0
    tp = target_profile["baseline_program_levels"] if line_features else np.zeros(len(PROGRAMS))
    tf = np.column_stack([tb[gi], np.full(len(gi), tk), np.full(len(gi), ki is None or not np.isfinite(tb[ki])),
                          np.full(len(gi), bool(target_profile.get("legacy_baseline", False))),
                          np.broadcast_to(tp, (len(gi), len(tp)))])
    sv, sf, masks, gv = [], [], [], []
    for c in sources:
        l, se = c.effects.get(kd, gi)
        sb = c.profile["baseline_log"]
        sk = sb[ki] if ki is not None and np.isfinite(sb[ki]) else 0.0
        own, _ = c.effects.get(kd, [ki]) if ki is not None else (np.array([np.nan]), None)
        valid = np.isfinite(l) & np.isfinite(sb[gi]) & np.isfinite(tb[gi])
        sh = shrink(l, se, cap=cap)
        similarity = centred_similarity(tb, sb) if line_features else 0.0
        pdiff = tp-c.profile["baseline_program_levels"] if line_features else np.zeros(len(PROGRAMS))
        cols = [np.clip(l, -cap, cap), np.clip(se, 0, cap), ~np.isfinite(se), sh,
                sb[gi], tb[gi]-sb[gi], np.full(len(gi), sk), np.full(len(gi), tk-sk),
                np.full(len(gi), np.clip(own[0], -cap, cap)), np.full(len(gi), not np.isfinite(own[0])),
                np.full(len(gi), c.effects.legacy),
                np.full(len(gi), similarity), np.zeros(len(gi)),
                np.full(len(gi), bool(c.profile.get("legacy_baseline", False))),
                np.broadcast_to(pdiff, (len(gi), len(tp)))]
        sf.append(np.nan_to_num(np.column_stack(cols), nan=0, posinf=cap, neginf=-cap))
        sv.append(np.nan_to_num(sh, nan=0)); masks.append(valid); gv.append(generics[c.name][gi])
    if not sources:
        raise ValueError("No permitted sources")
    sf = np.stack(sf, 1); sv = np.stack(sv, 1); mask = np.stack(masks, 1)
    for j in range(len(sources)):
        other = mask.copy(); other[:, j] = False
        votes = other & (np.abs(sv) > 0.05)
        same = np.sign(sv) == np.sign(sv[:, [j]])
        sf[:, j, SRC_NAMES.index("agreement")] = np.divide((votes & same).sum(1), votes.sum(1),
                              out=np.full(len(gi), 0.5), where=votes.sum(1) > 0)
    den = np.maximum(mask.sum(1), 1)
    generic = (np.stack(gv, 1)*mask).sum(1)/den
    return {"sv": sv.astype(np.float32), "sf": sf.astype(np.float32), "mask": mask,
            "tf": np.nan_to_num(tf).astype(np.float32), "generic": generic.astype(np.float32)}


def pad(f, n):
    d = n - f["sv"].shape[1]
    if d:
        f = dict(f)
        for key in ("sv", "mask"):
            f[key] = np.pad(f[key], ((0, 0), (0, d)))
        f["sf"] = np.pad(f["sf"], ((0, 0), (0, d), (0, 0)))
    return f


def pooled(f):
    m = f["mask"][:, :, None]
    n = np.maximum(m.sum(1), 1)
    mean = (f["sf"]*m).sum(1)/n
    # Center before squaring. E[x^2]-E[x]^2 in float32 spuriously produces
    # nonzero SD for a single source, which a scaler can amplify catastrophically.
    sd = np.sqrt((((f["sf"]-mean[:, None, :])**2)*m).sum(1)/n)
    base = (f["sv"]*f["mask"]).sum(1)/n[:, 0]
    inter_idx = [SRC_NAMES.index("baseline_difference"), SRC_NAMES.index("knockdown_gene_baseline_difference")]
    inter_idx += [SRC_NAMES.index("program_difference_" + p) for p in PROGRAMS]
    interactions = (f["sv"][:, :, None]*f["sf"][:, :, inter_idx]*m).sum(1)/n
    x = np.column_stack([mean, sd, f["tf"], interactions, f["generic"], n[:, 0]])
    return x.astype(np.float32), base


def sample_genes(eligible, sources, kd, rng, budget, signal_fraction, cap):
    """Uniform component + source-only enrichment, unique genes and no target labels."""
    if not 0 <= signal_fraction <= 1 or budget < 1:
        raise ValueError("Invalid gene sampling options")
    total = min(budget, len(eligible))
    n_signal = int(total*signal_fraction)
    uniform = rng.choice(eligible, total-n_signal, replace=False)
    if not n_signal:
        return uniform
    remaining = np.setdiff1d(eligible, uniform)
    evidence = np.zeros(len(remaining))
    for s in sources:
        lfc, se = s.effects.get(kd, remaining)
        evidence = np.maximum(evidence, np.nan_to_num(np.abs(shrink(lfc, se, cap=cap)), nan=0))
    probability = evidence + 0.05
    extra = rng.choice(remaining, n_signal, replace=False, p=probability/probability.sum())
    return np.concatenate([uniform, extra])


def balanced_label_weights(label_se, task, contexts, noise_floor=0.25, precision_cap=10.0, enabled=True):
    """Cap raw precision, then normalize WITHIN task before balancing backgrounds.

    This avoids awarding Orion more total weight just because its cell-level SE is
    smaller than donor SE. Missing/invalid SE gets its task's median valid precision;
    all-missing tasks receive uniform weights. Precision is not measurement harmonization.
    """
    if noise_floor <= 0 or precision_cap <= 0:
        raise ValueError("Label noise floor and precision cap must be positive")
    w = np.zeros(len(task), dtype=float)
    active = np.unique(task)
    states = {c.background: sum(contexts[i].background == c.background for i in active) for c in contexts}
    for ti in active:
        use = task == ti
        se = np.asarray(label_se[use], float)
        good = np.isfinite(se) & (se >= 0)
        precision = np.ones(len(se))
        if enabled and good.any():
            precision[good] = np.minimum(precision_cap, 1/(se[good]**2+noise_floor**2))
            precision[~good] = np.median(precision[good])
        w[use] = precision/precision.sum()/states[contexts[ti].background]
    return (w/w.mean()).astype(np.float32)


def build_rows(contexts, targets, genes, generics, seed=0, per_kd=128, dropout=0.25, cap=8,
               line_features=False, signal_fraction=0.5, label_noise_floor=0.25,
               label_weight_cap=10.0, label_weighting=True, max_source_count=None):
    blocks, labels, label_se, task = [], [], [], []
    smax = len({c.background for c in contexts})-1
    for ti, target in enumerate(contexts):
        for kd in targets:
            src = choose_sources([s for s in contexts if kd in s.effects.pos], target.background)
            if kd not in target.effects.pos or not src:
                continue
            rng = np.random.default_rng(seed_for(seed, target.name, kd))
            # Drop entire sources per KD BEFORE building generic/agreement/context features.
            keep = rng.random(len(src)) >= dropout
            if not keep.any():
                keep[rng.integers(len(src))] = True
            src = [s for s, yes in zip(src, keep) if yes]
            if max_source_count is not None and len(src) > max_source_count:
                take = sorted(rng.choice(len(src), max_source_count, replace=False))
                src = [src[i] for i in take]
            # Bulk eligibility here is a TRAINING screen, not the scorer's per-cell-mean gate.
            eligible = np.flatnonzero(np.isfinite(target.profile["baseline_cpm"]) & (target.profile["baseline_cpm"] > 5) & (genes != kd))
            gi = sample_genes(eligible, src, kd, rng, per_kd, signal_fraction, cap)
            y, ys = target.effects.get(kd, gi)
            f = features(target.profile, target.platform, src, kd, gi, genes, generics, cap, line_features)
            valid = np.isfinite(y) & f["mask"].any(1)
            # Match off-target scientific evaluation; own transcript remains an INPUT feature.
            valid &= genes[gi] != kd
            if valid.any():
                blocks.append(pad({k: v[valid] for k, v in f.items()}, smax))
                labels.append(y[valid]); label_se.append(ys[valid]); task.append(np.full(valid.sum(), ti))
    if not blocks:
        raise ValueError("No aligned overlapping training examples remain")
    f = {k: np.concatenate([b[k] for b in blocks]) for k in blocks[0]}
    y, t = np.concatenate(labels).astype(np.float32), np.concatenate(task)
    w = balanced_label_weights(np.concatenate(label_se), t, contexts, label_noise_floor, label_weight_cap, label_weighting)
    return f, y, w


def baseline_predictions(f):
    _, base = pooled(f)
    return {"zero": np.zeros(len(base)), "source_mean": base, "generic": f["generic"]}


def fit_ridge(f, y, w, val, alphas):
    x, base = pooled(f); xv, bv = pooled(val[0])
    scaler = StandardScaler().fit(x, sample_weight=w)
    active = scaler.var_ > 1e-8
    # Do not learn numerical noise in almost-constant features. Bound extrapolation
    # in standardized input space; this does not use held-out outcomes.
    x = np.clip(scaler.transform(x)*active, -5, 5)
    xv = np.clip(scaler.transform(xv)*active, -5, 5)
    candidates = []
    for alpha in alphas:
        m = Ridge(alpha=alpha).fit(x, y-base, sample_weight=w)
        pred = bv + m.predict(xv)
        loss = float(np.average(np.abs(pred-val[1]), weights=val[2]))
        candidates.append((loss, alpha, m))
    best = min(candidates, key=lambda v: v[0])
    return dict(kind="ridge", scaler=scaler, active=active, x_clip=5.0, estimator=best[2], alpha=best[1]), [dict(alpha=a, validation_mae=l) for l, a, _ in candidates]


def neural_class():
    import torch
    from torch import nn
    class Mixture(nn.Module):
        def __init__(self, residual=False):
            super().__init__()
            def mlp(d):
                return nn.Sequential(nn.Linear(d, 32), nn.GELU(), nn.Linear(32, 1))
            self.src = mlp(len(SRC_NAMES)+len(TGT_NAMES))
            self.gen = mlp(1+len(TGT_NAMES)); self.null = mlp(len(TGT_NAMES))
            self.res = mlp(len(SRC_NAMES)+len(TGT_NAMES)) if residual else None
            if self.res is not None:
                nn.init.zeros_(self.res[-1].weight); nn.init.zeros_(self.res[-1].bias)

        def forward(self, sv, sf, mask, tf, generic):
            t = tf[:, None, :].expand(-1, sv.shape[1], -1)
            score = self.src(torch.cat([sf, t], -1)).squeeze(-1).masked_fill(~mask, -1e9)
            weights = torch.softmax(torch.cat([score, self.gen(torch.cat([generic[:, None], tf], 1)), self.null(tf)], 1), 1)
            pred = (weights*torch.cat([sv, generic[:, None], torch.zeros_like(generic[:, None])], 1)).sum(1)
            correction = torch.zeros_like(pred)
            if self.res is not None:
                mean = (sf*mask[:, :, None]).sum(1)/mask.sum(1).clamp(min=1)[:, None]
                correction = self.res(torch.cat([mean, tf], 1)).squeeze(-1)
                pred += correction
            return pred, correction
    return Mixture


def neural_inputs(f, idx, scalers, device="cpu"):
    import torch
    sf = (f["sf"][idx]-scalers["sf"][0])/scalers["sf"][1]
    tf = (f["tf"][idx]-scalers["tf"][0])/scalers["tf"][1]
    return [torch.as_tensor(v, device=device) for v in (f["sv"][idx], sf.astype(np.float32),
            f["mask"][idx], tf.astype(np.float32), f["generic"][idx])]


def fit_neural(f, y, w, val, residual, seed, epochs=25):
    import torch
    torch.manual_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    norm = {}
    for key, arr in (("sf", f["sf"][f["mask"]]), ("tf", f["tf"])):
        sd = arr.std(0)
        # Constant training features carry no learned extrapolation signal. Suppress
        # their unseen deviations instead of feeding arbitrary untrained weights.
        norm[key] = (arr.mean(0), np.where(sd < 1e-6, np.inf, np.maximum(sd, 1e-3)))
    model = neural_class()(residual).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    best, state, stale, history = np.inf, None, 0, []
    rng = np.random.default_rng(seed)
    for epoch in range(epochs):
        model.train(); order = rng.permutation(len(y))
        for lo in range(0, len(order), 4096):
            ii = order[lo:lo+4096]
            pred, correction = model(*neural_inputs(f, ii, norm, device))
            yy, ww = torch.as_tensor(y[ii], device=device), torch.as_tensor(w[ii], device=device)
            loss = (ww*torch.nn.functional.huber_loss(pred, yy, reduction="none")).mean() + 0.1*correction.square().mean()
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval(); error, mass = 0., 0.
        with torch.no_grad():
            for lo in range(0, len(val[1]), 4096):
                ii = slice(lo, lo+4096)
                pred = model(*neural_inputs(val[0], ii, norm, device))[0].cpu().numpy()
                error += np.sum(val[2][ii]*np.abs(pred-val[1][ii])); mass += val[2][ii].sum()
        value = float(error/mass); history.append(dict(epoch=epoch, validation_mae=value))
        print(history[-1], flush=True)
        if value < best-1e-5:
            best, state, stale = value, {k: v.detach().cpu().numpy().copy() for k, v in model.state_dict().items()}, 0
        else:
            stale += 1
        if stale >= 5:
            break
    return dict(kind="mixture", residual=residual, state=state, norm=norm), history


def predict(model, f):
    if model["kind"] == "ridge":
        x, base = pooled(f)
        z = np.clip(model["scaler"].transform(x)*model["active"], -model["x_clip"], model["x_clip"])
        return base + model["estimator"].predict(z)
    import torch
    net = neural_class()(model["residual"])
    net.load_state_dict({k: torch.from_numpy(v) for k, v in model["state"].items()}); net.eval()
    with torch.no_grad():
        return net(*neural_inputs(f, slice(None), model["norm"]))[0].numpy()


def train(config_path, output, holdout=None, kind="ridge", allow_legacy=False, seed=0,
          max_kd=3000, per_kd=128, residual=False, epochs=25, line_features=None,
          signal_fraction=None, label_noise_floor=None, label_weight_cap=None,
          label_weighting=None, baseline_mode=None):
    cfg = json.loads(Path(config_path).read_text())
    genes = read_names(cfg["genes"], "gene_name")
    if holdout is not None and holdout not in {s["background"] for s in cfg["contexts"]}:
        raise ValueError("Unknown holdout background")
    # Config options also work with the existing outer-fold runner; explicit CLI wins.
    line_features = cfg.get("line_features", False) if line_features is None else line_features
    signal_fraction = cfg.get("signal_fraction", 0.5) if signal_fraction is None else signal_fraction
    label_noise_floor = cfg.get("label_noise_floor", 0.25) if label_noise_floor is None else label_noise_floor
    label_weight_cap = cfg.get("label_weight_cap", 10.0) if label_weight_cap is None else label_weight_cap
    label_weighting = cfg.get("label_weighting", True) if label_weighting is None else label_weighting
    baseline_mode = cfg.get("baseline_mode", "bulk") if baseline_mode is None else baseline_mode
    contexts = load_contexts(cfg, genes, [holdout] if holdout else [], allow_legacy, baseline_mode)
    if len({c.background for c in contexts}) < 2:
        raise ValueError("Need two remaining backgrounds to learn transfer")
    overlap = set()
    for c in contexts:
        other = set().union(*(set(s.effects.targets) for s in contexts if s.background != c.background))
        overlap |= set(c.effects.targets) & other
    kd = np.array(sorted(overlap))
    rng = np.random.default_rng(seed); rng.shuffle(kd)
    kd = kd[:max_kd] if max_kd else kd
    if len(kd) < 5:
        raise ValueError("Need at least five overlapping knockdowns")
    nv = max(1, int(len(kd)*0.2)); val_kd, tr_kd = kd[:nv], kd[nv:]
    cap = float(cfg.get("effect_cap", 8))
    generics = generic_signatures(contexts, tr_kd, genes, cap)
    row_options = dict(line_features=line_features, label_noise_floor=label_noise_floor,
                       label_weight_cap=label_weight_cap, label_weighting=label_weighting)
    f, y, w = build_rows(contexts, tr_kd, genes, generics, seed, per_kd, 0.25, cap,
                         signal_fraction=signal_fraction, **row_options)
    max_sources = int(f["mask"].sum(1).max())
    # No signal-enrichment in validation: preserve an unbiased gene sample.
    val = build_rows(contexts, val_kd, genes, generics, seed+1, per_kd, 0, cap,
                     signal_fraction=0, max_source_count=max_sources, **row_options)
    print(f"Training {len(y):,} rows; validating {len(val[1]):,}; backgrounds: {sorted({c.background for c in contexts})}", flush=True)
    if kind == "ridge":
        model, history = fit_ridge(f, y, w, val, cfg.get("ridge_alphas", [10, 100, 1000, 10000]))
    else:
        model, history = fit_neural(f, y, w, val, residual, seed, epochs)
    candidates = dict(baseline_predictions(val[0]), model=predict(model, val[0]))
    report = {k: float(np.average(np.abs(v-val[1]), weights=val[2])) for k, v in candidates.items()}
    artifact = dict(review_revision=1, model=model, genes=genes, generics=generics, cap=cap, seed=seed,
                    max_trained_sources=max_sources, line_features=line_features, baseline_mode=baseline_mode,
                    # Provenance declarations are metadata, not held-out expression outcomes.
                    baseline_declarations={str(Path(s.get("profile", str(Path(s["effects"])/"control_profile.npz"))).resolve()):
                                           s["baseline_convention"] for s in cfg["contexts"] if "baseline_convention" in s},
                    contexts=[c.spec for c in contexts], excluded_background=holdout,
                    train_targets=tr_kd.tolist(), inner_validation_targets=val_kd.tolist(),
                    allow_legacy=allow_legacy, feature_names=dict(source=SRC_NAMES, target=TGT_NAMES))
    out = Path(output); out.mkdir(parents=True, exist_ok=True)
    joblib.dump(artifact, out / "model.joblib", compress=3)
    save_json(out / "training_report.json", dict(inner_validation_mae=report,
              inner_validation_unweighted_mae={k: float(np.mean(np.abs(v-val[1]))) for k, v in candidates.items()}, history=history,
              max_trained_sources=max_sources, inference_policy="average trained-arity source subsets",
              line_features=line_features, baseline_mode=baseline_mode, platform_feature=False,
              signal_fraction=signal_fraction, validation_signal_fraction=0,
              label_weighting=label_weighting, label_noise_floor=label_noise_floor, label_weight_cap=label_weight_cap,
              train_backgrounds=sorted({c.background for c in contexts}), heldout_background=holdout,
              train_knockdowns=len(tr_kd), validation_knockdowns=len(val_kd), rows=len(y),
              legacy_contexts=[c.name for c in contexts if c.effects.legacy],
              note="Inner knockdown validation is for tuning only, not unseen-context evidence."))
    print(json.dumps(report, indent=2))
    return artifact


def subset_predict(model, profile, platform, sources, kd, gi, genes, generics,
                   cap, max_trained_sources, line_features=False):
    """Average all maximum-trained-size subsets, rebuilding features per subset.

    A fold trained on one source uses singleton averaging, not unseen two-source
    gates. A final model trained on two uses all three pairs of three available
    sources. Rows with no evidence in a subset do not vote for an artificial zero.
    """
    if max_trained_sources < 1:
        raise ValueError("Invalid saved training source count")
    size = min(len(sources), max_trained_sources)
    if size == 0:
        return np.zeros(len(gi)), 0
    total, votes = np.zeros(len(gi)), np.zeros(len(gi), int)
    n_subsets = 0
    for subset in combinations(sources, size):
        f = features(profile, platform, subset, kd, gi, genes, generics, cap, line_features)
        valid = f["mask"].any(1)
        if valid.any():
            total[valid] += predict(model, {k: v[valid] for k, v in f.items()})
            votes[valid] += 1
        n_subsets += 1
    return np.divide(total, votes, out=np.zeros_like(total), where=votes > 0), n_subsets


def infer(model_dir, profile_path, targets_csv, output, platform="unknown", chunk=2048):
    a = joblib.load(Path(model_dir)/"model.joblib")  # Load only your own trusted artifacts.
    if a.get("review_revision") != 1 or a.get("feature_names") != dict(source=SRC_NAMES, target=TGT_NAMES):
        raise ValueError("Retrain with the reviewed v3 script: this checkpoint uses the old feature/prediction schema")
    genes = a["genes"]; targets = read_names(targets_csv, "target_gene")
    profile = load_profile(profile_path, genes)
    declaration = a["baseline_declarations"].get(str(Path(profile_path).resolve()))
    sidecar = Path(str(profile_path)+".json")
    if declaration is None and sidecar.exists():
        declaration = json.loads(sidecar.read_text()).get("baseline_convention")
    profile = transfer_profile(profile, a["baseline_mode"], declaration, str(profile_path))
    contexts = [Context(s, genes, a["allow_legacy"], a["baseline_mode"]) for s in a["contexts"]]
    out = Path(output)
    if out.exists() and any(out.iterdir()):
        raise ValueError("Choose a new empty prediction directory")
    out.mkdir(parents=True, exist_ok=True)
    np.save(out/"genes.npy", genes); np.save(out/"targets.npy", targets)
    from numpy.lib.format import open_memmap
    preds = {name: open_memmap(out/f"{name}.npy", mode="w+", dtype="float32", shape=(len(targets), len(genes)))
             for name in ("model", "zero", "source_mean", "generic")}
    coverage = open_memmap(out/"source_count.npy", mode="w+", dtype="uint8", shape=(len(targets), len(genes)))
    diagnostics = []
    for i, kd in enumerate(targets):
        usable = choose_sources([c for c in contexts if kd in c.effects.pos])
        ncovered, n_subsets = 0, 0
        for lo in range(0, len(genes), chunk):
            gi = np.arange(lo, min(lo+chunk, len(genes)))
            if not usable:
                for arr in preds.values(): arr[i, gi] = 0
                coverage[i, gi] = 0
                continue
            f = features(profile, platform, usable, kd, gi, genes, a["generics"], a["cap"], a["line_features"])
            count = f["mask"].sum(1)
            yh, n_subsets = subset_predict(a["model"], profile, platform, usable, kd, gi, genes,
                                a["generics"], a["cap"], a["max_trained_sources"], a["line_features"])
            # Unlearned baselines deliberately retain ALL permitted sources.
            values = dict(baseline_predictions(f), model=yh)
            for name, v in values.items():
                preds[name][i, gi] = np.where(count > 0, np.clip(v, -a["cap"], a["cap"]), 0)
            coverage[i, gi] = count; ncovered += int((count > 0).sum())
        diagnostics.append(dict(target=kd, covered_genes=ncovered, source_names=[s.name for s in usable],
                                subset_size=min(len(usable), a["max_trained_sources"]), n_subsets=n_subsets))
    for arr in preds.values(): arr.flush()
    coverage.flush()
    save_json(out/"prediction_report.json", dict(model_dir=str(Path(model_dir).resolve()),
              target_profile=str(Path(profile_path).resolve()), platform=platform,
              platform_feature=False, baseline_mode=a["baseline_mode"], line_features=a["line_features"],
              max_trained_sources=a["max_trained_sources"], prediction_policy="mean over trained-arity subsets",
              heldout_background=a["excluded_background"], source_names=sorted({s for d in diagnostics for s in d["source_names"]}),
              output="log2_effect_estimate; unsupported genes receive explicit zero fallback", coverage=diagnostics))


def cosine_retrieval(predicted, real, block=64, tolerance=1e-6):
    """Effect-space retrieval with midranks: unique best match=1; all tied=0.5.

    Rows must use the same finite, off-panel gene axis and matching target order.
    Top-1 is fractional for ties. Not the official cell-eval2 count-space PDS.
    """
    p, r = np.asarray(predicted, np.float32), np.asarray(real, np.float32)
    if p.shape != r.shape or len(p) < 2 or p.shape[1] < 2 or not np.isfinite(p).all() or not np.isfinite(r).all():
        raise ValueError("Retrieval needs >=2 aligned targets and genes with finite values")
    p = p/np.maximum(np.linalg.norm(p, axis=1, keepdims=True), 1e-12)
    r = r/np.maximum(np.linalg.norm(r, axis=1, keepdims=True), 1e-12)
    rank, top1 = np.empty(len(p)), np.empty(len(p))
    for start in range(0, len(p), block):
        end = min(start+block, len(p))
        sim = p[start:end] @ r.T
        own = sim[np.arange(end-start), np.arange(start, end)]
        better = (sim > own[:, None]+tolerance).sum(1)
        tied = (np.abs(sim-own[:, None]) <= tolerance).sum(1)
        rank[start:end] = 1+better+0.5*(tied-1)
        top1[start:end] = np.where(better == 0, 1/np.maximum(tied, 1), 0)
    return dict(retrieval_rank=rank, retrieval_score=1-(rank-1)/(len(p)-1), retrieval_top1=top1)


def retrieval_tables(arrays, truth, profile, genes, targets, max_targets=512, seed=0):
    """A deterministic bounded panel avoids an accidental 18k x 18k x 18k job.

    For the challenge-size panel (e.g. 300 targets), every prediction competes against
    every truth. Use max_targets=0 for a larger exhaustive panel, at quadratic cost.
    The full REQUESTED target-gene panel is excluded even when subsampling targets.
    """
    if max_targets < 0:
        raise ValueError("retrieval_max_targets must be >=0")
    ii = np.array([i for i, k in enumerate(targets) if k in truth.pos], dtype=int)
    available = len(ii)
    if max_targets and len(ii) > max_targets:
        rng = np.random.default_rng(seed)
        ii = np.sort(rng.choice(ii, max_targets, replace=False))
    eligible = (profile["mean_cpm"] > 5) & ~np.isin(genes, targets)
    gi = np.flatnonzero(eligible)
    info = dict(space="log2_effect_cosine_proxy_NOT_official_PDS", n_available_targets=available,
                max_targets=max_targets, seed=seed, requested_panel_targets=len(targets),
                sampled=available > len(ii), target_gene_exclusion="entire requested panel")
    if len(ii) < 2 or len(gi) < 2:
        return pd.DataFrame(), dict(info, status="insufficient targets or off-panel genes")
    real = np.stack([truth.get(targets[i], gi)[0] for i in ii]).astype(np.float32)
    # Do not shrink the common gene axis to nothing because of an entirely missing row.
    usable = np.isfinite(real).any(1)
    ii, real = ii[usable], real[usable]
    common = np.isfinite(real).all(0)
    for arr in arrays.values():
        for start in range(0, len(ii), 64):
            common &= np.isfinite(arr[np.ix_(ii[start:start+64], gi)]).all(0)
    gi, real = gi[common], real[:, common]
    info.update(n_targets=len(ii), n_genes=len(gi), targets=targets[ii].tolist())
    if len(ii) < 2 or len(gi) < 2:
        return pd.DataFrame(), dict(info, status="insufficient common finite genes or measured targets")
    rows = []
    for name, arr in arrays.items():
        result = cosine_retrieval(arr[np.ix_(ii, gi)], real)
        rows.append(pd.DataFrame(dict(target=targets[ii], method=name, **result)))
    return pd.concat(rows, ignore_index=True), dict(info, status="ok")


def evaluate_tables(pred_dir, truth_dir, profile_path, output, allow_legacy=False,
                    retrieval_max_targets=512, retrieval_seed=0):
    p = Path(pred_dir); genes = np.load(p/"genes.npy"); targets = np.load(p/"targets.npy")
    truth = Effects(truth_dir, genes, allow_legacy)
    profile = load_profile(profile_path, genes)
    arrays = {n: np.load(p/f"{n}.npy", mmap_mode="r") for n in ("model", "zero", "source_mean", "generic")}
    rows = []
    for i, kd in enumerate(targets):
        y, _ = truth.get(kd, np.arange(len(genes)))
        valid = np.isfinite(y) & (profile["mean_cpm"] > 5) & (genes != kd)
        if not valid.any(): continue
        for name, arr in arrays.items():
            yy, pp = y[valid], np.asarray(arr[i, valid], float)
            corr = float(np.corrcoef(yy, pp)[0, 1]) if yy.std() > 0 and pp.std() > 0 else np.nan
            rows.append(dict(target=kd, method=name, n_genes=int(valid.sum()),
                             mae=float(np.mean(np.abs(yy-pp))), mse=float(np.mean((yy-pp)**2)), correlation=corr))
    frame = pd.DataFrame(rows)
    if frame.empty: raise ValueError("No truth-aligned targets to evaluate")
    Path(output).mkdir(parents=True, exist_ok=True)
    retrieval, retrieval_info = retrieval_tables(arrays, truth, profile, genes, targets,
                                                 retrieval_max_targets, retrieval_seed)
    metric_columns = ["mae", "mse", "correlation"]
    if not retrieval.empty:
        extra = ["retrieval_rank", "retrieval_score", "retrieval_top1"]
        frame = frame.merge(retrieval, on=["target", "method"], how="left", validate="one_to_one")
        retrieval.to_csv(Path(output)/"retrieval_per_knockdown.csv", index=False)
        retrieval.groupby("method")[extra].mean().to_csv(Path(output)/"retrieval_summary.csv")
        metric_columns += extra
    save_json(Path(output)/"retrieval_metadata.json", retrieval_info)
    frame.to_csv(Path(output)/"per_knockdown.csv", index=False)
    frame.groupby("method")[metric_columns].mean().to_csv(Path(output)/"summary.csv")
    save_json(Path(output)/"limitations.json", dict(legacy_truth=truth.legacy,
              note="Table-only off-target diagnostics and effect-space retrieval. Not cell-eval2/Wilcoxon/official PDS. "
                   "Compare retrieval and MAE jointly across held-out backgrounds; inner tuning still uses weighted MAE. "
                   "Legacy mean_cpm screening is approximate if its baseline was ratio-of-sums."))
    print(frame.groupby("method")[metric_columns].mean())


if __name__ == "__main__":
    p = argparse.ArgumentParser(description=__doc__); sub = p.add_subparsers(dest="command", required=True)
    t = sub.add_parser("train"); t.add_argument("--config", required=True); t.add_argument("--out", required=True)
    t.add_argument("--holdout"); t.add_argument("--kind", choices=["ridge", "mixture"], default="ridge")
    t.add_argument("--allow-legacy-effects", action="store_true"); t.add_argument("--seed", type=int, default=0)
    t.add_argument("--max-knockdowns", type=int, default=3000, help="0 = all overlapping knockdowns")
    t.add_argument("--genes-per-knockdown", type=int, default=128); t.add_argument("--residual", action="store_true")
    t.add_argument("--epochs", type=int, default=25)
    t.add_argument("--line-features", action=argparse.BooleanOptionalAction, default=None,
                   help="Ablation: enable global program levels/differences and centred context Pearson (default off)")
    t.add_argument("--signal-fraction", type=float, default=None, help="Source-enriched share of training gene samples (default 0.5; 0 = uniform)")
    t.add_argument("--label-noise-floor", type=float, default=None, help="SE floor in log2 units (default 0.25)")
    t.add_argument("--label-weight-cap", type=float, default=None, help="Cap inverse variance before within-task normalization (default 10)")
    t.add_argument("--label-weighting", action=argparse.BooleanOptionalAction, default=None)
    t.add_argument("--baseline-mode", choices=["bulk", "per_cell"], default=None, help="Context estimator, default bulk; effect labels unchanged")
    i = sub.add_parser("predict"); i.add_argument("--model", required=True); i.add_argument("--profile", required=True)
    i.add_argument("--targets", required=True); i.add_argument("--out", required=True); i.add_argument("--platform", default="unknown")
    e = sub.add_parser("evaluate-tables"); e.add_argument("--predictions", required=True); e.add_argument("--truth", required=True)
    e.add_argument("--profile", required=True); e.add_argument("--out", required=True); e.add_argument("--allow-legacy-effects", action="store_true")
    e.add_argument("--retrieval-max-targets", type=int, default=512, help="0 = exhaustive; >512-target panels are otherwise deterministically sampled")
    e.add_argument("--retrieval-seed", type=int, default=0)
    a = p.parse_args()
    if a.command == "train":
        train(a.config, a.out, a.holdout, a.kind, a.allow_legacy_effects, a.seed, a.max_knockdowns,
              a.genes_per_knockdown, a.residual, a.epochs, a.line_features, a.signal_fraction,
              a.label_noise_floor, a.label_weight_cap, a.label_weighting, a.baseline_mode)
    elif a.command == "predict": infer(a.model, a.profile, a.targets, a.out, a.platform)
    else: evaluate_tables(a.predictions, a.truth, a.profile, a.out, a.allow_legacy_effects,
                          a.retrieval_max_targets, a.retrieval_seed)

#### vcc_generate_v3.py

In [ ]:
%%writefile ~/vcc/code/vcc_generate_v3.py
"""VCC 2026 cell generator v3: predicted fold changes -> 400 raw-count cells per knockdown, plus the oracle test.

Two commands
  submit : deltas for each line (from vcc_transfer_model_v3 `predict`, or any .npz) -> calibrated -> 400 cells per
           target per line, streamed into one submission .h5ad (same layout as M1, so `vcc prep` works unchanged),
           plus answer-free checks.
  oracle : on a context WITH real knockdown cells (Orion HCT116/HEK293T target cells, Arc H1), feed the generator the
           TRUE fold changes, generate cells from that context's own controls, and score them against the real
           knockdown cells. This measures the generator alone: any gap here is a gap no transfer model can fix.

Defaults reproduce M1's generator exactly (so the oracle can compare every switch against M1):
  0 calibrate : delta <- alpha*delta; clip +-4; |delta| < 0.01 -> 0; genes <= 5 CPM in the line's controls -> 0
  1 target    : the knocked-down gene's factor = residual measured in the sources (clipped 0.05-0.9), never the model's
  3 base cells: 400 controls drawn evenly across the non-targeting guides, without replacement
  5 counts    : decreases = binomial thinning; increases = + Poisson((f-1) * expected); expected = lib_cell * share_g
Switches (all OFF by default; turn on only if the oracle shows they help):
  --composition changed : if a knockdown's net share change |sum s_g (2^d_g - 1)| > --comp-thr, absorb it in the
                          changed genes so untouched genes keep their CPM (avoids spurious DE on untouched genes)
  --pi P                : fraction of escaper cells (unchanged); the rest get f=(2^d-P)/(1-P) so the mean still matches
  --up-w W              : increases proportional to each cell's own count (W=1) vs baseline expectation (W=0, M1)
  --match-pool          : correct delta for the 400 sampled cells' own deviation from the line mean (removes our
                          sampling noise from the pseudobulk)
  --tilt                : resample controls by cell-cycle score so the predicted cycling shift moves the cell
                          distribution, not only the mean (implies --match-pool)

Colab examples
  !python vcc_generate_v3.py submit --data "$DATA" --lines A B C \
      --delta A=/content/drive/MyDrive/vcc_lib/pred_A --delta B=... --delta C=... \
      --kd-orion /content/drive/MyDrive/vcc/orion/HCT116_pseudobulk.h5ad /content/drive/MyDrive/vcc/orion/HEK293T_pseudobulk.h5ad \
      --kd-tcell /content/drive/MyDrive/vcc_lib/tcell/effects/tcell_targets_effects.h5ad --out /content/vcc_m2
  !python vcc_generate_v3.py oracle --cells /content/drive/MyDrive/vcc/orion/HCT116_cells_targets.h5ad \
      --genes "$DATA/gene_names.csv" --out /content/oracle_hct116 --label m1_defaults
  (re-run oracle with switches, e.g. --pi 0.15 --label pi015; results append to oracle_summary.csv)
"""
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import mannwhitneyu

try:
    from anndata.io import read_elem, write_elem, sparse_dataset
except ImportError:  # older anndata
    from anndata.experimental import read_elem, write_elem, sparse_dataset

CONTROL = "non-targeting"
CC_GENES = ("MCM5 PCNA TYMS FEN1 MCM2 MCM4 RRM1 UNG GINS2 MCM6 CDCA7 DTL PRIM1 UHRF1 HELLS RFC2 RPA2 NASP RAD51AP1 GMNN "
            "WDR76 SLBP CCNE2 UBR7 POLD3 MSH2 ATAD2 RAD51 RRM2 CDC45 CDC6 EXO1 TIPIN DSCC1 BLM CASP8AP2 USP1 CLSPN POLA1 "
            "CHAF1B BRIP1 E2F8 HMGB2 CDK1 NUSAP1 UBE2C BIRC5 TPX2 TOP2A NDC80 CKS2 NUF2 CKS1B MKI67 TMPO CENPF TACC3 SMC4 "
            "CCNB2 CKAP2L CKAP2 AURKB BUB1 KIF11 ANP32E TUBB4B GTSE1 KIF20B HJURP CDCA3 CDC20 TTK CDC25C KIF2C RANGAP1 "
            "NCAPD2 DLGAP5 CDCA2 CDCA8 ECT2 KIF23 HMMR AURKA PSRC1 ANLN LBR CKAP5 CENPE CTCF NEK2 G2E3 GAS2L3 CBX5 CENPA").split()


class Cfg:
    def __init__(self, a):
        self.alpha = getattr(a, "alpha", 1.0); self.max_lfc = 4.0; self.min_lfc = 0.01
        self.kd_clip = (0.05, 0.9); self.default_kd = 0.30; self.n_cells = getattr(a, "n_cells", 400)
        self.composition = getattr(a, "composition", "none"); self.comp_thr = getattr(a, "comp_thr", 0.02)
        self.pi = getattr(a, "pi", 0.0); self.up_w = getattr(a, "up_w", 0.0)
        self.tilt = getattr(a, "tilt", False); self.match_pool = getattr(a, "match_pool", False) or self.tilt
        self.seed = getattr(a, "seed", 2026)

    def as_dict(self):
        return {k: v for k, v in vars(self).items()}


# ---------------------------------------------------------------- inputs
def read_list(path, col):
    return pd.read_csv(path)[col].astype(str).tolist()


def aligned_X(a, genes):
    """cells x our genes, CSR float32 (missing genes = 0, extra genes dropped)."""
    X = a.X.tocsr() if sp.issparse(a.X) else sp.csr_matrix(a.X)
    names = a.var_names.astype(str).tolist()
    if names == list(genes):
        return X.astype(np.float32)
    pos = {g: i for i, g in enumerate(genes)}
    src = np.array([i for i, g in enumerate(names) if g in pos]); dst = np.array([pos[names[i]] for i in src])
    X = X[:, src].tocoo()
    print(f"  re-indexed genes: {len(src):,}/{len(genes):,} of ours present")
    return sp.csr_matrix((X.data.astype(np.float32), (X.row, dst[X.col])), shape=(X.shape[0], len(genes)))


class Line:
    """Control cells of one context plus the statistics the generator needs."""
    def __init__(self, X, guide, genes):
        self.X = X.tocsr().astype(np.float32); self.genes = genes; self.n = X.shape[0]
        self.lib = np.asarray(self.X.sum(1)).ravel()
        self.cpm_cells = sp.diags(1e6 / np.maximum(self.lib, 1)) @ self.X
        self.cpm = np.asarray(self.cpm_cells.mean(0)).ravel()           # mean per-cell CPM (scorer gate uses > 5)
        self.share = np.asarray(self.X.sum(0)).ravel() / self.X.sum()   # pooled share of molecules
        self.gate = self.cpm > 5
        g = np.asarray(guide).astype(str) if guide is not None else np.zeros(self.n, str)
        self.groups = [np.flatnonzero(g == u) for u in np.unique(g)]
        gp = {x: i for i, x in enumerate(genes)}
        self.cc_idx = np.array([gp[x] for x in CC_GENES if x in gp and self.cpm[gp[x]] > 1], int)
        if len(self.cc_idx) >= 5:
            L = self.cpm_cells[:, self.cc_idx].toarray()
            self.cyc = np.log2(L + 1).mean(1)                            # per-cell cycling level (log2 units)
        else:
            self.cyc = None

    @staticmethod
    def from_h5ad(path, genes, control_col=None):
        a = ad.read_h5ad(path)
        if control_col and control_col in a.obs:
            a = a[a.obs[control_col].astype(str).values == CONTROL].copy()
        guide = next((a.obs[c].values for c in ("ntc_id", "guide_id", "sgID_AB", "guide") if c in a.obs), None)
        return Line(aligned_X(a, genes), guide, genes)


def load_deltas(path, targets, genes, array="model"):
    """v3 prediction dir (genes.npy, targets.npy, <array>.npy) or .npz (delta [+targets, genes]) -> T x G, NaN -> 0."""
    p = Path(path)
    if p.is_dir():
        D = np.load(p / f"{array}.npy", mmap_mode="r"); t = np.load(p / "targets.npy").astype(str)
        g = np.load(p / "genes.npy").astype(str)
    else:
        z = np.load(p, allow_pickle=False); D = z["delta"]
        t = z["targets"].astype(str) if "targets" in z else np.array(targets)
        g = z["genes"].astype(str) if "genes" in z else np.array(genes)
    out = np.zeros((len(targets), len(genes)), np.float32)
    tp = {x: i for i, x in enumerate(t)}; gp = {x: i for i, x in enumerate(g)}
    gsrc = np.array([gp.get(x, -1) for x in genes]); ok = gsrc >= 0
    missing = 0
    for i, x in enumerate(targets):
        if x in tp:
            out[i, ok] = np.asarray(D[tp[x]])[gsrc[ok]]
        else:
            missing += 1
    if missing:
        print(f"  WARNING {path}: {missing} targets missing -> no change predicted for them")
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def kd_from_sources(targets, genes, orion_paths=(), tcell_path=None, tcell_condition="Stim48hr", pseudo=0.5):
    """Knocked-down gene's own log2 drop per target, averaged across sources (M1 rule). NaN if no source has it."""
    gp = {g: i for i, g in enumerate(genes)}; vals = [[] for _ in targets]
    for path in orion_paths:
        with h5py.File(path, "r") as f:
            obs = read_elem(f["obs"]); names = list(obs.index.astype(str)); n = obs["n_cells"].values.astype(float)
            var = read_elem(f["var"]); vg = list(var.index.astype(str))
            vpos = {g: i for i, g in enumerate(vg)}
            c = names.index(CONTROL)
            want = [(i, t) for i, t in enumerate(targets) if t in names and t in vpos]
            if not want:
                continue
            rows = sorted({c} | {names.index(t) for _, t in want})
            S = f["layers/sum_cpm"][rows, :]; rpos = {r: k for k, r in enumerate(rows)}
            ctrl = S[rpos[c]] / n[c]
            for i, t in want:
                r = names.index(t); col = vpos[t]
                vals[i].append(np.log2((S[rpos[r], col] / n[r] + pseudo) / (ctrl[col] + pseudo)))
    if tcell_path:
        t = ad.read_h5ad(tcell_path)
        if "condition" in t.obs:
            t = t[t.obs["condition"].astype(str) == tcell_condition]
        tg = t.obs["target_gene"].astype(str).values; tv = {g: i for i, g in enumerate(t.var_names.astype(str))}
        X = np.asarray(t.X)
        for i, x in enumerate(targets):
            r = np.flatnonzero(tg == x)
            if len(r) and x in tv and np.isfinite(X[r[0], tv[x]]):
                vals[i].append(float(X[r[0], tv[x]]))
    return np.array([np.mean(v) if v else np.nan for v in vals])


def kd_residual(log2_drop, cfg):
    r = np.where(np.isfinite(log2_drop), 2.0 ** np.nan_to_num(log2_drop), cfg.default_kd)
    return np.clip(r, *cfg.kd_clip)


# ---------------------------------------------------------------- core generator
def calibrate(D, line, targets, cfg):
    """Steps 0-1: alpha, clip, floor, 5-CPM mask; zero the knocked-down gene (its factor is handled separately)."""
    D = np.clip(cfg.alpha * D, -cfg.max_lfc, cfg.max_lfc).astype(np.float32)
    D[np.abs(D) < cfg.min_lfc] = 0.0
    D[:, ~line.gate] = 0.0
    gp = {g: i for i, g in enumerate(line.genes)}
    for i, t in enumerate(targets):
        if t in gp:
            D[i, gp[t]] = 0.0
    return D


def pick_cells(line, d, cfg, rng):
    """Step 3 (+ optional tilt): which control cells become this knockdown's 400 cells."""
    n = cfg.n_cells
    if cfg.tilt and line.cyc is not None:
        dcc = float(np.mean(d[line.cc_idx]))                       # predicted cycling shift (log2)
        v = line.cyc - line.cyc.mean(); s = max(v.std(), 1e-6)
        if abs(dcc) > 0.02:
            lo, hi = -30.0, 30.0
            for _ in range(50):                                     # bisection: weighted mean shift == dcc
                b = (lo + hi) / 2; w = np.exp(np.clip(b * v / s, -50, 50)); m = (w * v).sum() / w.sum()
                lo, hi = (b, hi) if m < dcc else (lo, b)
            w = np.exp(np.clip(b * v / s, -50, 50)); w /= w.sum()
            k = min(n, int((w > 0).sum()))
            return rng.choice(line.n, k, replace=False, p=w) if k == n else rng.choice(line.n, n, replace=True, p=w)
    per = int(np.ceil(n / len(line.groups)))
    pick = np.concatenate([rng.choice(v, size=min(per, len(v)), replace=False) for v in line.groups])
    if len(pick) >= n:
        return rng.choice(pick, n, replace=False)
    rest = np.setdiff1d(np.arange(line.n), pick)
    return np.concatenate([pick, rng.choice(rest, n - len(pick), replace=len(rest) < n - len(pick))])


def generate_one(line, d, j, kd, cfg, rng, pick=None):
    """One knockdown: returns (CSR counts n_cells x G, diagnostics). d = calibrated delta row; j = target gene index."""
    d = d.copy(); diag = {}
    if pick is None:
        pick = pick_cells(line, d, cfg, rng)
    B = line.X[pick].toarray()
    if cfg.match_pool:                                              # remove our own sampling deviation from the mean
        pool = np.asarray(line.cpm_cells[pick].mean(0)).ravel()
        dev = np.log2((pool + 0.5) / (line.cpm + 0.5))
        m = line.gate.copy()
        if j is not None:
            m[j] = False
        d[m] = np.clip(d[m] - dev[m], -cfg.max_lfc, cfg.max_lfc)
        d[np.abs(d) < cfg.min_lfc] = 0.0
    cols = np.flatnonzero(d != 0)
    if j is not None:
        cols = np.union1d(cols, [j])
    fac = 2.0 ** d[cols].astype(np.float64)
    jpos = None
    if j is not None:
        jpos = int(np.searchsorted(cols, j)); fac[jpos] = kd
    s = line.share[cols]; net = float((s * (fac - 1)).sum()); diag["net_share_change"] = net
    if cfg.composition == "changed" and abs(net) > cfg.comp_thr:   # absorb net change in changed non-target genes
        other = np.ones(len(cols), bool)
        if jpos is not None:
            other[jpos] = False
        A = float((s[other] * fac[other]).sum())
        if A - net > 0 and other.any():
            fac[other] *= (A - net) / A; diag["composition_factor"] = (A - net) / A
    diag["n_changed"] = int(len(cols))
    if len(cols):
        sub = B[:, cols].astype(np.int64)
        if cfg.pi > 0:                                              # escapers keep control-like expression
            esc = rng.random(len(pick)) < cfg.pi
            f_in = np.maximum((fac - cfg.pi) / (1 - cfg.pi), 0.0)
            F = np.where(esc[:, None], 1.0, f_in[None, :])
        else:
            F = np.broadcast_to(fac[None, :], sub.shape)
        sub = rng.binomial(sub, np.minimum(F, 1.0))                 # decreases (p = 1 leaves counts unchanged)
        upf = np.maximum(F - 1.0, 0.0)
        if upf.any():
            base = line.lib[pick][:, None] * line.share[cols][None, :]
            expected = cfg.up_w * B[:, cols] + (1 - cfg.up_w) * base
            sub = sub + rng.poisson(upf * expected)
        B[:, cols] = sub.astype(np.float32)
    out = sp.csr_matrix(B); out.eliminate_zeros()
    return out, diag, d


# ---------------------------------------------------------------- local DE (scorer-like: per-cell CPM, gate, BH 0.05)
def cpm_dense(X, idx):
    X = X.tocsr(); lib = np.asarray(X.sum(1)).ravel()
    return (sp.diags(1e6 / np.maximum(lib, 1)) @ X)[:, idx].toarray().astype(np.float32)


def bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    out = np.empty(n); out[o] = np.minimum(q, 1); return out


def de_set(cells_cpm, ctrl_cpm, alpha=0.05):
    """Wilcoxon per gene (cells vs controls), BH. Returns boolean significant mask over the given gene columns."""
    var = (cells_cpm.std(0) > 0) | (ctrl_cpm.std(0) > 0)
    sig = np.zeros(cells_cpm.shape[1], bool)
    if var.any():
        p = mannwhitneyu(cells_cpm[:, var], ctrl_cpm[:, var], axis=0, method="asymptotic").pvalue
        sig[np.flatnonzero(var)] = bh(np.nan_to_num(p, nan=1.0)) < alpha
    return sig


def answer_free_check(cells, line, d_planned, j, ctrl_cpm, gate_idx):
    """Realised vs planned effect, spurious DE on untouched genes, NMAE-gate pass (>= 10 significant genes)."""
    pool = np.asarray((sp.diags(1e6 / np.maximum(np.asarray(cells.sum(1)).ravel(), 1)) @ cells).mean(0)).ravel()
    real = np.log2((pool + 0.5) / (line.cpm + 0.5))
    ch = (d_planned != 0) & line.gate; un = (d_planned == 0) & line.gate
    if j is not None:
        ch[j] = un[j] = False
    sig = de_set(cpm_dense(cells, gate_idx), ctrl_cpm)
    gpos = np.zeros(len(line.genes), bool); gpos[gate_idx] = True
    sig_full = np.zeros(len(line.genes), bool); sig_full[gate_idx] = sig
    if j is not None:
        sig_full[j] = False
    return dict(realised_vs_planned_corr=float(np.corrcoef(real[ch], d_planned[ch])[0, 1]) if ch.sum() > 2 else np.nan,
                realised_minus_planned_mae=float(np.abs(real[ch] - d_planned[ch]).mean()) if ch.any() else np.nan,
                untouched_mean_shift=float(real[un].mean()), n_sig=int(sig_full.sum()),
                n_sig_untouched=int((sig_full & un).sum()), passes_nmae_gate=bool(sig_full.sum() >= 10))


# ---------------------------------------------------------------- submit
def cmd_submit(a):
    cfg = Cfg(a); data = Path(a.data)
    genes = read_list(data / "gene_names.csv", "gene_name"); targets = read_list(data / "pert_counts.csv", "target_gene")
    gp = {g: i for i, g in enumerate(genes)}
    deltas = dict(x.split("=", 1) for x in a.delta)
    missing = [L for L in a.lines if L not in deltas]
    if missing:
        raise SystemExit(f"no --delta for lines {missing}")
    log2kd = kd_from_sources(targets, genes, a.kd_orion or (), a.kd_tcell, a.kd_tcell_condition)
    kd = kd_residual(log2kd, cfg)
    print(f"target residual: {np.isfinite(log2kd).sum()}/{len(targets)} measured in sources, median {np.median(kd):.2f}")
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True); H5 = out / "prediction.h5ad"
    obs = pd.concat([pd.DataFrame({"target_gene": t, "context": L}, index=[f"{L}_{t}_{i:03d}" for i in range(cfg.n_cells)])
                     for L in a.lines for t in targets])
    obs["target_gene"] = obs["target_gene"].astype("category"); obs["context"] = obs["context"].astype("category")
    var = pd.DataFrame(index=pd.Index(genes, name="gene_name"))
    report = dict(settings=cfg.as_dict(), deltas=deltas, delta_array=a.delta_array, lines={})
    rng0 = np.random.default_rng(cfg.seed)
    with h5py.File(H5, "w") as f:
        write_elem(f, "obs", obs); write_elem(f, "var", var)
        f.attrs["encoding-type"] = "anndata"; f.attrs["encoding-version"] = "0.1.0"
        X_ds = None
        for L in a.lines:
            t0 = time.time()
            line = Line.from_h5ad(data / f"context_{L}.h5ad", genes)
            D = calibrate(load_deltas(deltas[L], targets, genes, a.delta_array), line, targets, cfg)
            rng = np.random.default_rng([cfg.seed, ord(L[0])])
            chk_t = set(rng0.choice(len(targets), min(a.checks, len(targets)), replace=False).tolist()) if a.checks else set()
            gate_idx = np.flatnonzero(line.gate)
            ctrl_cpm = cpm_dense(line.X[rng0.choice(line.n, min(4000, line.n), replace=False)], gate_idx) if chk_t else None
            diags, checks = [], []
            for ti, t in enumerate(targets):
                j = gp.get(t)
                cells, dg, d_used = generate_one(line, D[ti], j, kd[ti], cfg, rng)
                diags.append(dg)
                if ti in chk_t:
                    checks.append(dict(target=t, **answer_free_check(cells, line, d_used, j, ctrl_cpm, gate_idx)))
                if X_ds is None:
                    write_elem(f, "X", cells); X_ds = sparse_dataset(f["X"])
                else:
                    X_ds.append(cells)
            net = np.array([x["net_share_change"] for x in diags]); nch = np.array([x["n_changed"] for x in diags])
            rep = dict(scored_genes=int(line.gate.sum()), genes_changed_median=float(np.median(nch)),
                       net_share_change_abs_p90=float(np.percentile(np.abs(net), 90)),
                       knockdowns_net_over_thr=int((np.abs(net) > cfg.comp_thr).sum()),
                       target_residual_median=float(np.median(kd)), seconds=round(time.time() - t0))
            if checks:
                c = pd.DataFrame(checks); c.to_csv(out / f"checks_{L}.csv", index=False)
                rep["checks"] = dict(n=len(c), realised_corr_median=float(c.realised_vs_planned_corr.median()),
                                     n_sig_median=float(c.n_sig.median()), n_sig_untouched_median=float(c.n_sig_untouched.median()),
                                     frac_pass_nmae_gate=float(c.passes_nmae_gate.mean()))
            report["lines"][L] = rep
            print(f"line {L}: {json.dumps(rep)}", flush=True)
            del line
    p = ad.read_h5ad(H5, backed="r")
    assert p.shape == (len(a.lines) * len(targets) * cfg.n_cells, len(genes)), p.shape
    x = p.X[:2000]; x = x.toarray() if sp.issparse(x) else x
    report["integer_counts"] = bool(np.all(np.mod(x, 1) == 0)); p.file.close()
    json.dump(report, open(out / "generator_report.json", "w"), indent=1, default=float)
    print(f"WROTE {H5} ({os.path.getsize(H5) / 1e9:.2f} GB); integer counts: {report['integer_counts']}")
    print(f'next: vcc prep {H5} -g "$DATA/gene_names.csv" --perts "$DATA/pert_counts.csv" -o {out}/prediction.vcc')


# ---------------------------------------------------------------- oracle
def bulk_lognorm(X):
    s = np.asarray(X.sum(0)).ravel(); return np.log1p(s / max(s.sum(), 1) * 1e6)


def score_sets(pred, real, ctrl_cpm, gate_idx, genes, panel_idx):
    """pred/real: {target: CSR cells}. Approximations of the scorer's metrics (NOT cell-eval2)."""
    T = list(real); gp = {g: i for i, g in enumerate(genes)}
    bp = {t: bulk_lognorm(pred[t]) for t in T}; br = {t: bulk_lognorm(real[t]) for t in T}
    keep_pds = np.ones(len(genes), bool); keep_pds[panel_idx] = False
    rows = []
    for t in T:
        j = gp.get(t); m = np.ones(len(genes), bool)
        if j is not None:
            m[j] = False
        dist = np.array([np.abs(bp[t][keep_pds] - br[u][keep_pds]).mean() for u in T])
        rank = (dist < dist[T.index(t)]).sum() / max(len(T) - 1, 1)
        sp_ = de_set(cpm_dense(pred[t], gate_idx), ctrl_cpm); sr = de_set(cpm_dense(real[t], gate_idx), ctrl_cpm)
        if j is not None and j in set(gate_idx):
            k = int(np.flatnonzero(gate_idx == j)[0]); sp_[k] = sr[k] = False
        inter, union = (sp_ & sr).sum(), (sp_ | sr).sum()
        rows.append(dict(target=t, mse=float(((bp[t] - br[t])[m] ** 2).mean()), pds_rank=float(rank),
                         jaccard=float(inter / union) if union else np.nan, n_sig_pred=int(sp_.sum()), n_sig_real=int(sr.sum()),
                         real_gate10=bool(sr.sum() >= 10), n_cells_real=int(real[t].shape[0])))
    return pd.DataFrame(rows)


def cmd_oracle(a):
    cfg = Cfg(a); genes = read_list(a.genes, "gene_name"); gp = {g: i for i, g in enumerate(genes)}
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    h = ad.read_h5ad(a.cells); lab = h.obs[a.obs_col].astype(str).values
    X = aligned_X(h, genes)
    guide = next((h.obs[c].values for c in ("ntc_id", "guide_id", "sgID_AB", "guide") if c in h.obs), None)
    ci = np.flatnonzero(lab == CONTROL)
    line = Line(X[ci], None if guide is None else np.asarray(guide)[ci], genes)
    vc = pd.Series(lab[lab != CONTROL]).value_counts()
    tlist = sorted(vc[vc >= a.min_cells].index)
    rng = np.random.default_rng(cfg.seed)
    if a.n_targets and len(tlist) > a.n_targets:
        tlist = sorted(rng.choice(tlist, a.n_targets, replace=False).tolist())
    print(f"oracle: {len(ci):,} controls, {len(tlist)} targets (>= {a.min_cells} cells), label '{a.label}'")
    # Split each target's real cells: half 1 estimates the 'true' effect, half 2 is the truth we score against.
    # (Estimating and scoring on the same cells would reward fitting their noise.)
    split_rng = np.random.default_rng([cfg.seed, 7])
    ctrl_var = np.maximum(np.asarray(line.cpm_cells.multiply(line.cpm_cells).mean(0)).ravel() - line.cpm ** 2, 0) / line.n
    real, est, true_d, kd, esc = {}, {}, [], [], []
    for t in tlist:
        idx = split_rng.permutation(np.flatnonzero(lab == t)); k = len(idx) // 2
        R = X[idx[:k]]; est[t] = R; real[t] = X[idx[k:]]
        Rc = sp.diags(1e6 / np.maximum(np.asarray(R.sum(1)).ravel(), 1)) @ R
        cp = np.asarray(Rc.mean(0)).ravel()
        d = np.log2((cp + 0.5) / (line.cpm + 0.5))
        if a.oracle_tau > 0:                                        # same SNR shrink as M1 (noisy 'truth' -> ~0)
            vk = np.maximum(np.asarray(Rc.multiply(Rc).mean(0)).ravel() - cp ** 2, 0) / R.shape[0]
            se = np.sqrt(vk / (cp + 0.5) ** 2 + ctrl_var / (line.cpm + 0.5) ** 2) / np.log(2)
            snr2 = np.where(se > 0, (d / np.maximum(se, 1e-12)) ** 2, 0.0)
            j0 = gp.get(t); dj = d[j0] if j0 is not None else None
            d = d * snr2 / (snr2 + a.oracle_tau)
            if j0 is not None:
                d[j0] = dj                                          # target's own drop stays unshrunk (it is the kd)
        true_d.append(d)
        j = gp.get(t)
        kd.append(np.clip(2.0 ** d[j], *cfg.kd_clip) if j is not None else cfg.default_kd)
        if j is not None and line.cpm[j] > 20:                     # escaper upper bound: 2 x fraction above control median
            med = np.median(np.asarray(line.cpm_cells[:, j].todense()).ravel())
            kc = np.asarray((sp.diags(1e6 / np.maximum(np.asarray(R.sum(1)).ravel(), 1)) @ R)[:, j].todense()).ravel()
            esc.append(min(1.0, 2 * float((kc > med).mean())))
    D = calibrate(np.array(true_d, np.float32), line, tlist, cfg)
    gate_idx = np.flatnonzero(line.gate); panel = [gp[t] for t in tlist if t in gp]
    ctrl_cpm = cpm_dense(line.X[rng.choice(line.n, min(4000, line.n), replace=False)], gate_idx)
    gen = {}
    for i, t in enumerate(tlist):
        c = cfg.n_cells
        cfg.n_cells = real[t].shape[0] if a.match_n else c        # same cell count as truth -> same DE power
        gen[t], _, _ = generate_one(line, D[i], gp.get(t), kd[i], cfg, rng)
        cfg.n_cells = c
    res = score_sets(gen, real, ctrl_cpm, gate_idx, genes, panel); res["variant"] = a.label
    anchors = out / "oracle_anchors.csv"
    if not anchors.exists():                                        # computed once per output dir
        print("computing anchors (context-mean baseline = 0, half-vs-half replicate = 1) ...")
        mean_d = np.tile(D.mean(0), (len(tlist), 1))
        base = {t: generate_one(line, mean_d[i], gp.get(t), kd[i], Cfg(argparse.Namespace(n_cells=real[t].shape[0])), rng)[0]
                for i, t in enumerate(tlist)}
        b = score_sets(base, real, ctrl_cpm, gate_idx, genes, panel); b["variant"] = "anchor_context_mean"
        r = score_sets(est, real, ctrl_cpm, gate_idx, genes, panel); r["variant"] = "anchor_replicate_halves"
        pd.concat([b, r]).to_csv(anchors, index=False)
    res.to_csv(out / f"oracle_{a.label}.csv", index=False)
    allres = pd.concat([pd.read_csv(anchors)] + [pd.read_csv(q) for q in sorted(out.glob("oracle_*.csv"))
                                                 if q.name not in ("oracle_anchors.csv", "oracle_summary.csv")])
    summ = allres.groupby("variant").agg(mse=("mse", "mean"), pds_rank=("pds_rank", "mean"), jaccard=("jaccard", "mean"),
                                         n_sig_pred=("n_sig_pred", "median"), n_sig_real=("n_sig_real", "median"))
    summ.to_csv(out / "oracle_summary.csv")
    info = dict(label=a.label, settings=cfg.as_dict(), n_targets=len(tlist), match_n=a.match_n, oracle_tau=a.oracle_tau,
                escape_upper_bound_median=float(np.median(esc)) if esc else None,
                note="Approximate metrics, not cell-eval2. True effects from one half of the real cells; all variants and anchors scored on the other half.")
    json.dump(info, open(out / f"oracle_{a.label}.json", "w"), indent=1, default=float)
    print(summ.round(4).to_string()); print(json.dumps({k: info[k] for k in ("n_targets", "escape_upper_bound_median")}))
    if a.write_h5ad:                                                # for running the real cell-eval2 on the same cells
        def pack(dct, ctrl=True):
            mats = list(dct.values()) + ([line.X] if ctrl else [])
            labs = np.concatenate([[t] * dct[t].shape[0] for t in dct] + ([[CONTROL] * line.n] if ctrl else []))
            return ad.AnnData(sp.vstack(mats).tocsr(), obs=pd.DataFrame({"target_gene": labs}, index=[str(i) for i in range(len(labs))]),
                              var=pd.DataFrame(index=pd.Index(genes, name="gene_name")))
        pack(gen).write_h5ad(out / f"pred_{a.label}.h5ad"); pack(real).write_h5ad(out / "real.h5ad")
        print("wrote pred/real h5ad for cell-eval2 (see `cell-eval run --help` for the vcc2026 profile flags)")


def add_switches(p):
    p.add_argument("--alpha", type=float, default=1.0); p.add_argument("--n-cells", type=int, default=400)
    p.add_argument("--composition", choices=["none", "changed"], default="none")
    p.add_argument("--comp-thr", type=float, default=0.02); p.add_argument("--pi", type=float, default=0.0)
    p.add_argument("--up-w", type=float, default=0.0); p.add_argument("--match-pool", action="store_true")
    p.add_argument("--tilt", action="store_true"); p.add_argument("--seed", type=int, default=2026)


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    s = sub.add_parser("submit"); add_switches(s)
    s.add_argument("--data", required=True, help="folder with gene_names.csv, pert_counts.csv, context_<L>.h5ad")
    s.add_argument("--lines", nargs="+", default=["A", "B", "C"])
    s.add_argument("--delta", action="append", required=True, help="L=path (v3 prediction dir or .npz); repeat per line")
    s.add_argument("--delta-array", default="model", help="for v3 dirs: model | source_mean | generic")
    s.add_argument("--kd-orion", nargs="*"); s.add_argument("--kd-tcell"); s.add_argument("--kd-tcell-condition", default="Stim48hr")
    s.add_argument("--checks", type=int, default=20, help="answer-free checks on this many targets per line (0 = off)")
    s.add_argument("--out", required=True)
    o = sub.add_parser("oracle"); add_switches(o)
    o.add_argument("--cells", required=True, help="h5ad with real knockdown cells AND controls (raw counts)")
    o.add_argument("--genes", required=True); o.add_argument("--obs-col", default="target_gene")
    o.add_argument("--min-cells", type=int, default=100); o.add_argument("--n-targets", type=int, default=40)
    o.add_argument("--match-n", action=argparse.BooleanOptionalAction, default=True)
    o.add_argument("--oracle-tau", type=float, default=4.0, help="SNR shrink of the estimated true effects (0 = off)")
    o.add_argument("--label", default="m1_defaults"); o.add_argument("--write-h5ad", action="store_true")
    o.add_argument("--out", required=True)
    a = ap.parse_args()
    cmd_submit(a) if a.cmd == "submit" else cmd_oracle(a)

## 2. Run the pipeline
Long steps run in the background: run the cell, then re-run its `status(...)` cell until it says **FINISHED OK**.
Following your notes, Arc H1 is **not** in training. It is used as an independent test of the final model (step 2.3) and to tune the generator (step 2.5).

### 2.1 Effect tables from the public data + contexts.json (background, a few minutes)

In [ ]:
bg('effects', r"""
rm -rf $LIB/effects/ORION_HCT116 $LIB/effects/ORION_HEK293T $LIB/effects/TCELL_Stim48hr $LIB/effects/TCELL_Rest $LIB/effects/ARC_H1
"$PY" vcc_build_effects_v3.py orion --pseudobulk $LIB/orion/HCT116_pseudobulk.h5ad  --genes $DATA/gene_names.csv --name ORION_HCT116  --background HCT116  --out $LIB/effects/ORION_HCT116
"$PY" vcc_build_effects_v3.py orion --pseudobulk $LIB/orion/HEK293T_pseudobulk.h5ad --genes $DATA/gene_names.csv --name ORION_HEK293T --background HEK293T --out $LIB/effects/ORION_HEK293T
"$PY" vcc_build_effects_v3.py tcell --effects $LIB/tcell/TCELL_Stim48hr_effects.h5ad --genes $DATA/gene_names.csv --name TCELL_Stim48hr --priority 1 --out $LIB/effects/TCELL_Stim48hr
"$PY" vcc_build_effects_v3.py tcell --effects $LIB/tcell/TCELL_Rest_effects.h5ad     --genes $DATA/gene_names.csv --name TCELL_Rest --out $LIB/effects/TCELL_Rest
"$PY" vcc_build_effects_v3.py h1    --pseudobulk $LIB/h1/h1_pseudobulk.h5ad          --genes $DATA/gene_names.csv --name ARC_H1 --out $LIB/effects/ARC_H1
"$PY" vcc_build_effects_v3.py make-config --genes $DATA/gene_names.csv --out $LIB/contexts.json \
  --dirs $LIB/effects/ORION_HCT116 $LIB/effects/ORION_HEK293T $LIB/effects/TCELL_Stim48hr $LIB/effects/TCELL_Rest
""")

In [ ]:
status('effects')

### 2.2 Profiles of the challenge lines (runs here, a few minutes)

In [ ]:
run(r"""
mkdir -p $LIB/profiles
for L in A B C; do
  "$PY" vcc_context_profile_v3.py --counts $DATA/context_$L.h5ad --all-controls --genes $DATA/gene_names.csv --out $LIB/profiles/$L.npz
done
ls -lh $LIB/profiles
""")

### 2.3 Hold-out tests + final model (background; the go/no-go decision)
For each public background, train without it and predict it. Then train the final model on everything and test it on Arc H1, which was never used in training.

In [ ]:
bg('holdout', r"""
declare -A TRUTH=([CD4_T]=TCELL_Stim48hr [HCT116]=ORION_HCT116 [HEK293T]=ORION_HEK293T)
for BG in "${!TRUTH[@]}"; do CTX=${TRUTH[$BG]}; rm -rf $LIB/eval/$BG $LIB/models/holdout_$BG
  "$PY" vcc_transfer_model_v3.py train --config $LIB/contexts.json --holdout $BG --out $LIB/models/holdout_$BG
  "$PY" vcc_transfer_model_v3.py predict --model $LIB/models/holdout_$BG --profile $LIB/effects/$CTX/control_profile.npz --targets $DATA/pert_counts.csv --out $LIB/eval/$BG/pred
  "$PY" vcc_transfer_model_v3.py evaluate-tables --predictions $LIB/eval/$BG/pred --truth $LIB/effects/$CTX --profile $LIB/effects/$CTX/control_profile.npz --out $LIB/eval/$BG
done
rm -rf $LIB/models/final $LIB/eval/H1
"$PY" vcc_transfer_model_v3.py train --config $LIB/contexts.json --out $LIB/models/final
"$PY" vcc_transfer_model_v3.py predict --model $LIB/models/final --profile $LIB/effects/ARC_H1/control_profile.npz --targets $DATA/pert_counts.csv --out $LIB/eval/H1/pred
"$PY" vcc_transfer_model_v3.py evaluate-tables --predictions $LIB/eval/H1/pred --truth $LIB/effects/ARC_H1 --profile $LIB/effects/ARC_H1/control_profile.npz --out $LIB/eval/H1
""")

In [ ]:
status('holdout')

In [ ]:
run('for f in $LIB/eval/*/summary.csv; do echo "== $f"; cat "$f"; done')      # paste this output to Claude

### 2.4 Predictions for lines A, B, C (runs here)

In [ ]:
run(r"""
for L in A B C; do rm -rf $LIB/pred/$L
  "$PY" vcc_transfer_model_v3.py predict --model $LIB/models/final --profile $LIB/profiles/$L.npz --targets $DATA/pert_counts.csv --out $LIB/pred/$L
done
""")

### 2.5 Generator oracle on Arc H1 (background; picks the generator switches)

In [ ]:
bg('oracle', r"""
for V in "--label m1_defaults" "--match-pool --label matchpool" "--up-w 1 --label upw1" "--pi 0.15 --label pi015"; do
  "$PY" vcc_generate_v3.py oracle --cells $LIB/h1/h1_cells_oracle.h5ad --genes $DATA/gene_names.csv --out $LIB/oracle_h1 $V
done
cat $LIB/oracle_h1/oracle_summary.csv
""")

In [ ]:
status('oracle', n=40)      # paste the summary table to Claude

### 2.6 Generate the submission (background)
Set `SWITCHES` to the generator options the oracle picked. Set `DELTA_ARRAY = 'source_mean'` if step 2.3 said no-go.

In [ ]:
RUN_NAME = 'm2'                 # results go to ~/vcc/results/<RUN_NAME>
SWITCHES = ''                   # e.g. '--match-pool'
DELTA_ARRAY = 'model'           # model | source_mean | generic
bg('submit', rf"""
rm -rf ~/vcc/results/{RUN_NAME}
"$PY" vcc_generate_v3.py submit --data $DATA --lines A B C --delta-array {DELTA_ARRAY} {SWITCHES} \
  --delta A=$LIB/pred/A --delta B=$LIB/pred/B --delta C=$LIB/pred/C \
  --kd-orion $LIB/orion/HCT116_pseudobulk.h5ad $LIB/orion/HEK293T_pseudobulk.h5ad \
  --kd-tcell $LIB/tcell/tcell_targets_effects.h5ad --out ~/vcc/results/{RUN_NAME}
""")

In [ ]:
status('submit')

### 2.7 Package + submit to the leaderboard (asks for your VCC API key; it is not saved)

In [ ]:
import getpass
os.environ['VCC_TOKEN'] = getpass.getpass('VCC API key: ')
MESSAGE = f'{RUN_NAME}: v3 transfer + generator {SWITCHES}'.strip()
run(rf"""
R=~/vcc/results/{RUN_NAME}
vcc prep $R/prediction.h5ad -g $DATA/gene_names.csv --perts $DATA/pert_counts.csv -o $R/prediction.vcc
vcc submit $R/prediction.vcc -m {shlex.quote(MESSAGE)} --wait
""")

### 2.8 Back up results and this notebook to the bucket (end of every session)

In [ ]:
run(r"""
gcloud storage rsync -r $LIB $B/lib --exclude='.*\.h5ad$'
gcloud storage rsync -r ~/vcc/results $B/results
gcloud storage cp ~/vcc/*.ipynb $B/code/
""")